# HerbariumScribe v2 — image → layout → OCR → RAG → LLM → reconciled DwC record

**Pipeline for the NHM SCRIBE / DiSSCo UK use case.** Built on the Dillen et al. (2019) Zenodo benchmark; honest about its limits.

## What changed from v1.4

1. **Data leakage fixed.** A disjoint `DEMO_SET` (RAG few-shots) and `EVAL_SET` (evaluation) are produced by stratified random sampling on `institutionCode`, with `random_state=42`. Wherever the dataset allows, the two sets share no institutions.
2. **Heuristic crops replaced** by a three-tier label region detector: **Hespi** (Turnbull et al. 2024, herbarium-trained YOLOv8) → **PaddleOCR PPStructure** (PubLayNet-trained, document layout) → geometric crops. The tier that fired is logged per record and reported in the final summary.
3. **OCR upgraded** from Tesseract-only to **PaddleOCR (`latin` script)** as primary, with Tesseract as a second-opinion fallback when Paddle confidence is low or the engine fails. The engine choice is logged per crop.
4. **Real RAG**, not TF-IDF prompt augmentation. A `sentence-transformers/all-MiniLM-L6-v2` index is built over (a) field-guide docs, (b) the DEMO_SET examples, (c) a fixed European-flora **GBIF accepted-species subset**, (d) a **GeoNames** country + admin1 + cities15000 subset. Retrieval is per-source, top-k blended into the LLM prompt.
5. **Post-hoc reconciliation** through the **GBIF `/species/match` API** as the primary tool for `scientificName`, with cosine over the local subset as fallback. Both `scientificName_verbatim` and `scientificName_canonical` are stored; synonyms are flagged, never silently overwritten. Countries/admin1 reconciled via `pycountry` + GeoNames.
6. **Honest, stratified evaluation** on **N=50** records, with metrics split by **OCR quality tertile** (low/med/high) and by **field**. Coverage of the gold genera by the fixed GBIF subset is reported. Bootstrap 95% CIs are shown on the headline numbers because N=50 is still small.
7. **Architecture cleaned up.** Every stage writes a CSV/JSONL checkpoint so partial reruns work. The LLM is behind a single `call_llm(messages) -> str` so the backend (Qwen-local default, Anthropic API fallback) flips with one config flag and an env var.

## Honesty notes (read before defending numbers)

- The "gold" we evaluate against is the **curator-normalised Darwin Core record from the Dillen 2019 benchmark, not full label transcription.** A perfect extraction of `"G. Forrest"` will still match a gold value of `"George Forrest"` only after our string normalisation — so we are measuring *normalised structured extraction*, not OCR CER/WER. CER/WER claims require manual transcription ground truth that this benchmark does not provide.
- "OCR proxy" metrics (whether gold field values appear inside OCR text) are reported **separately** from extraction metrics, and labelled as a proxy for transcription quality — not as a transcription score.
- N=50 with stratification gives us small per-cell denominators (often 5–15). The CIs reflect this honestly; do not over-interpret per-field deltas between methods unless they exceed the CI band.

## Architecture

```mermaid
flowchart TD
    A[Dillen 2019<br/>Zenodo metadata] --> B[Stratified split<br/>DEMO_SET / EVAL_SET<br/>by institutionCode]
    B --> C[Image download<br/>with retries + cache]
    C --> D[Layout detection<br/>Hespi → PPStructure → geometric]
    D --> E[OCR per region<br/>PaddleOCR latin → Tesseract fallback]
    E --> F[OCR proxy metrics<br/>literal + normalised]

    G1[GBIF accepted species<br/>fixed European-flora families] --> H[Sentence-transformer<br/>all-MiniLM-L6-v2 index]
    G2[GeoNames country + admin1<br/>+ cities15000] --> H
    G3[Field-guide docs<br/>+ DEMO_SET examples] --> H

    E --> L[rule_ocr<br/>regex baseline]
    E --> M[qwen_no_rag<br/>LLM on OCR only]
    E --> N[qwen_rag_v2<br/>LLM + real retrieval]
    H --> N

    L --> O[Reconciliation<br/>GBIF /species/match<br/>+ pycountry + GeoNames cosine fallback]
    M --> O
    N --> O

    O --> P[Stratified evaluation<br/>by field × OCR tertile<br/>bootstrap CIs]
    P --> Q[Graph export<br/>+ collector entity resolution]
    P --> R[Final markdown report]
```


In [ ]:
# ============================================================
# 1. Install dependencies
# ============================================================
# Design notes on deps:
#   - paddleocr 2.7 + paddlepaddle-gpu 2.6 is a known-good combo on Colab T4.
#     We pin to avoid surprise breakage; PaddleOCR install on Colab is the
#     single biggest install-time risk in this pipeline.
#   - hespi pulls in ultralytics, torchapp, and TrOCR. We try it first; if the
#     install fails we fall through to PPStructure (which we get for free
#     with paddleocr) and finally geometric crops.
#   - bitsandbytes is only needed for the Qwen-4bit backend. If you flip
#     LLM_BACKEND='anthropic' you don't need it, but it's small enough to
#     install unconditionally.

import subprocess, sys, os

def pip_install(args, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    cmd += args
    return subprocess.run(cmd, check=False).returncode

# Apt: Tesseract for the fallback OCR path.
os.system("apt-get -qq update")
os.system("apt-get -qq install -y tesseract-ocr")

# Core scientific + utility stack.
pip_install([
    "-U",
    "pandas", "numpy", "requests", "pillow", "opencv-python-headless",
    "pytesseract", "rapidfuzz", "networkx", "scikit-learn", "pycountry",
])

# LLM stack (Qwen local + Anthropic API fallback).
pip_install([
    "-U",
    "transformers", "accelerate", "bitsandbytes",
    "sentence-transformers",
    "anthropic",
])

# OCR + layout (PaddleOCR). Pinned because newer versions have broken Colab
# in the past. If this fails, the OCR stage will fall through to Tesseract.
PADDLE_OK = pip_install(["paddlepaddle-gpu==2.6.1", "paddleocr==2.7.0.3"]) == 0
print("PaddleOCR install ok:", PADDLE_OK)

# Hespi (herbarium-specific YOLOv8 sheet-component detector).
# If this fails, layout detection falls through to PPStructure / geometric.
HESPI_OK = pip_install(["hespi"]) == 0
print("Hespi install ok:", HESPI_OK)

# Imports — done here so any missing dep is visible up front.
from pathlib import Path
import re, json, time, shutil, random, warnings, gc, io, zipfile, hashlib
from typing import Optional, Tuple, List, Dict, Any
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
from PIL import Image
import cv2
import pytesseract
import networkx as nx
import pycountry

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
# ============================================================
# 2. Configuration
# ============================================================
# Single source of truth for run knobs. Reduce N_EVAL to 10 for debug runs.

# Sample sizes
N_EVAL = 50          # eval-set size; spec target is 50
N_DEMO = 4           # few-shot examples for RAG; must be disjoint from EVAL
RANDOM_SEED = 42

# LLM backend: "qwen" (default, runs locally on T4) or "anthropic" (API fallback).
# Flip with one variable. Anthropic key is read from env, never hardcoded.
LLM_BACKEND = "qwen"                                  # "qwen" | "anthropic"
QWEN_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
QWEN_USE_4BIT = True
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-haiku-4-5")
# To use Anthropic: set LLM_BACKEND='anthropic' AND set env var ANTHROPIC_API_KEY.

# Layout detector tier order: hespi > ppstructure > geometric.
# Set USE_HESPI=False to skip Hespi entirely (faster debug, less accurate).
USE_HESPI = HESPI_OK
USE_PPSTRUCTURE = PADDLE_OK

# OCR: PaddleOCR primary in 'latin' script (covers en/fr/de/nl/it/es Latin-alphabet
# labels in Dillen 2019). Tesseract used as second opinion when Paddle confidence
# is below this threshold or Paddle errors.
OCR_PRIMARY_LANG = "latin"
PADDLE_LOW_CONF_THRESHOLD = 0.50
TESS_CONFIGS = ["--oem 3 --psm 6", "--oem 3 --psm 11"]

# Reconciliation thresholds (sentence-transformer cosine over local GBIF subset).
# These are the **fallback** path; the primary path is GBIF /species/match API.
# Calibrated as reasoned starting points; the report prints the actual cosine
# distribution per run so you can defend or retune them.
COS_AUTO_REPLACE = 0.92   # >=: auto-replace with canonical (verbatim still stored)
COS_FLAG_REVIEW  = 0.75   # 0.75..0.92: flag for review, keep verbatim
                          # <0.75: keep LLM output unchanged, demote confidence

# Fixed European-flora plant families for the GBIF subset. This is the
# Q1(c) choice: declare the index in advance, then report coverage against
# the eval set. Picked from standard European herbarium composition surveys.
EUROPEAN_FLORA_FAMILIES = [
    "Asteraceae", "Fabaceae", "Poaceae", "Rosaceae", "Orchidaceae",
    "Lamiaceae", "Brassicaceae", "Apiaceae", "Ranunculaceae", "Ericaceae",
    "Caryophyllaceae", "Scrophulariaceae", "Boraginaceae", "Liliaceae",
    "Cyperaceae", "Polygonaceae", "Plantaginaceae", "Salicaceae", "Fagaceae",
    "Betulaceae",
]
GBIF_SPECIES_PER_FAMILY = 400   # cap; ~8k names total, fits memory + API budget

# Source metadata
ZENODO_CSV_URL = "https://zenodo.org/records/2566726/files/oo_244128.csv?download=1"

# Darwin Core fields we touch.
TARGET_FIELDS = [
    "occurrenceID", "catalogNumber", "institutionCode", "scientificName",
    "recordedBy", "eventDate", "country", "stateProvince", "verbatimLocality",
    "decimalLatitude", "decimalLongitude", "typeStatus", "jpegURL",
]
EXTRACTION_FIELDS = [f for f in TARGET_FIELDS if f != "jpegURL"]
EVAL_FIELDS = [
    "catalogNumber", "institutionCode", "scientificName", "recordedBy",
    "eventDate", "country", "stateProvince", "verbatimLocality",
    "decimalLatitude", "decimalLongitude", "typeStatus",
]

# Project layout. Every stage writes here so partial reruns are cheap.
ROOT = Path("/content/herbarium_scribe_v2")
RAW_DIR       = ROOT / "data" / "raw"
IMG_DIR       = RAW_DIR / "images"
CROP_DIR      = RAW_DIR / "crops"
PROCESSED_DIR = ROOT / "data" / "processed"
AUTHORITY_DIR = ROOT / "data" / "authority"
EMBED_DIR     = ROOT / "data" / "embeddings"
REPORT_DIR    = ROOT / "reports"
LOG_DIR       = ROOT / "logs"
for d in [RAW_DIR, IMG_DIR, CROP_DIR, PROCESSED_DIR, AUTHORITY_DIR,
          EMBED_DIR, REPORT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Centralised stage log: every non-trivial event appends one JSONL row here.
# This is what we grep in the final report to compute fallback rates etc.
STAGE_LOG_PATH = LOG_DIR / "stage_events.jsonl"
def log_event(stage: str, record_id: str = "", level: str = "info", **kwargs):
    row = {"ts": time.time(), "stage": stage, "record_id": record_id, "level": level, **kwargs}
    with STAGE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

print("N_EVAL:", N_EVAL, " N_DEMO:", N_DEMO)
print("LLM backend:", LLM_BACKEND, "(Qwen:", QWEN_MODEL_ID, "| API:", ANTHROPIC_MODEL, ")")
print("Use Hespi:", USE_HESPI, " | Use PPStructure:", USE_PPSTRUCTURE)
print("Project root:", ROOT)


In [ ]:
# ============================================================
# 3. Load Dillen 2019 metadata and map to Darwin Core fields
# ============================================================
# Logic is essentially v1.4's, kept stable so the v1.4-vs-v2 diff stays focused
# on the upgraded stages. The honest comment: aliasing on column-substring is
# imperfect; if the Zenodo CSV schema changes, this needs revisiting.

def clean_str(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x).strip()
    return "" if s.lower() in {"nan", "none", "null"} else s

def make_fallback_records():
    """Tiny synthetic set so the notebook still demos when Zenodo is unreachable."""
    return pd.DataFrame([
        {"occurrenceID": "urn:demo:specimen:1", "catalogNumber": "E00633257",
         "institutionCode": "E", "scientificName": "Abelia forrestii (Diels) W.W.Sm.",
         "recordedBy": "George Forrest", "eventDate": "1915",
         "country": "China", "stateProvince": "Yunnan",
         "verbatimLocality": "Yunnan, West China",
         "decimalLatitude": "", "decimalLongitude": "",
         "typeStatus": "", "jpegURL": ""},
    ])

try:
    raw_df = pd.read_csv(ZENODO_CSV_URL, dtype=str, on_bad_lines="skip", low_memory=False)
    data_source = "Zenodo metadata (Dillen et al. 2019, record 2566726)"
    raw_df.to_csv(RAW_DIR / "zenodo_metadata.csv", index=False)
except Exception as e:
    log_event("metadata_load", level="error", error=repr(e))
    raw_df = make_fallback_records()
    data_source = "FALLBACK (Zenodo unreachable)"

ALIASES = {
    "occurrenceID":     ["occurrenceID", "occurrenceId", "id", "gbifID", "persistentID"],
    "catalogNumber":    ["catalogNumber", "catalogueNumber", "catalogue", "barcode", "specimenNumber"],
    "institutionCode":  ["institutionCode", "institution", "ownerInstitutionCode", "collectionCode"],
    "scientificName":   ["scientificName", "acceptedScientificName", "taxonName", "species", "name"],
    "recordedBy":       ["recordedBy", "collector", "collectors", "leg"],
    "eventDate":        ["eventDate", "verbatimEventDate", "dateIdentified", "collectionDate"],
    "country":          ["country", "countryCode", "verbatimCountry"],
    "stateProvince":    ["stateProvince", "province", "state", "county"],
    "verbatimLocality": ["verbatimLocality", "locality", "localityText", "location"],
    "decimalLatitude":  ["decimalLatitude", "lat", "latitude"],
    "decimalLongitude": ["decimalLongitude", "lon", "lng", "longitude"],
    "typeStatus":       ["typeStatus", "typestatus", "nomenclaturalTypeStatus"],
    "jpegURL":          ["jpegURL", "jpegURI", "jpgURL", "imageURL", "imageURI"],
}

def find_col(df, aliases):
    lower = {c.lower(): c for c in df.columns}
    for a in aliases:
        if a.lower() in lower:
            return lower[a.lower()]
    for c in df.columns:
        cl = c.lower()
        for a in aliases:
            if a.lower() in cl:
                return c
    return None

field_map = {field: find_col(raw_df, aliases) for field, aliases in ALIASES.items()}

# Build a DwC-shaped DataFrame (1:1 row preservation).
dc = pd.DataFrame()
for field in TARGET_FIELDS:
    col = field_map.get(field)
    dc[field] = raw_df[col].fillna("").astype(str) if col else ""

# Quality filter — keep records with image URL + at least 3 of the core
# evaluable fields. This is the v1.4 filter, kept so the sampling pool is
# comparable. Then we sample from this pool; no head() bias.
core_cols = ["scientificName", "catalogNumber", "institutionCode", "country"]
info_count = (dc[core_cols].replace("", np.nan).notna()).sum(axis=1)
has_image  = dc["jpegURL"].apply(clean_str) != ""
candidate_pool = dc.loc[(info_count >= 3) & has_image].reset_index(drop=True).copy()

print("Data source:", data_source)
print("Raw rows:", raw_df.shape[0], "| Candidate pool:", candidate_pool.shape[0])
print("Field map:")
for k, v in field_map.items():
    print(f"  {k:18s} -> {v}")
candidate_pool.to_csv(PROCESSED_DIR / "candidate_pool.csv", index=False)
display(candidate_pool.head(2))


In [ ]:
# ============================================================
# 4. Stratified DEMO_SET / EVAL_SET split  (fixes v1.4 leakage)
# ============================================================
# Two changes vs v1.4:
#   (a) DEMO_SET and EVAL_SET are disjoint. The 4 RAG few-shot records
#       never appear in evaluation.
#   (b) Sampling is stratified random on institutionCode with seed=42,
#       not head(N). The Zenodo CSV is institution-ordered; head(N) gives
#       a single-institution slice.
#
# We additionally prefer an INSTITUTION-DISJOINT split when feasible:
# the few-shot examples should not leak institutional formatting
# conventions (handwriting register, barcode prefix style) into the
# eval set. If the data does not allow this, we fall back to a
# within-institution split and print a loud warning.

def stratified_sample(df: pd.DataFrame, n: int, by: str, seed: int = 42) -> pd.DataFrame:
    """
    Proportional stratified sample by `by` column. Strata with too few rows
    are taken whole. Top-up from largest stratum if rounding leaves us short.
    """
    if len(df) <= n:
        return df.copy()
    counts = df[by].fillna("").value_counts()
    proportions = counts / counts.sum()
    take = (proportions * n).round().astype(int).to_dict()
    # Adjust to hit exactly n
    while sum(take.values()) > n:
        biggest = max(take, key=take.get)
        take[biggest] -= 1
    while sum(take.values()) < n:
        biggest = max(counts.index, key=lambda k: counts[k] - take.get(k, 0))
        take[biggest] = take.get(biggest, 0) + 1
    out = []
    rng = np.random.RandomState(seed)
    for stratum, k in take.items():
        if k <= 0:
            continue
        sub = df[df[by].fillna("") == stratum]
        k = min(k, len(sub))
        out.append(sub.sample(k, random_state=rng.randint(0, 1_000_000)))
    return pd.concat(out).sample(frac=1.0, random_state=seed).reset_index(drop=True)

# Step 1: sample EVAL_SET stratified by institutionCode.
eval_set = stratified_sample(candidate_pool, n=N_EVAL, by="institutionCode", seed=RANDOM_SEED)

# Step 2: try institution-disjoint DEMO_SET.
remaining = candidate_pool.drop(index=candidate_pool.index[
    candidate_pool["occurrenceID"].isin(eval_set["occurrenceID"])
])
eval_insts = set(eval_set["institutionCode"])
disjoint_pool = remaining[~remaining["institutionCode"].isin(eval_insts)]

demo_overlap_warning = None
if len(disjoint_pool) >= N_DEMO:
    # Try to spread DEMO across as many non-eval institutions as possible.
    demo_set = stratified_sample(disjoint_pool, n=N_DEMO, by="institutionCode", seed=RANDOM_SEED)
else:
    demo_set = remaining.sample(min(N_DEMO, len(remaining)), random_state=RANDOM_SEED)
    demo_overlap_warning = (
        f"WARNING: only {len(disjoint_pool)} institution-disjoint records available "
        f"for DEMO_SET (need {N_DEMO}). Falling back to within-institution sampling. "
        f"This means DEMO few-shots may leak institutional formatting conventions "
        f"to EVAL records from the same institution."
    )

# Assign record IDs and persist.
demo_set = demo_set.reset_index(drop=True)
demo_set["record_id"] = [f"demo_{i:04d}" for i in range(len(demo_set))]
eval_set = eval_set.reset_index(drop=True)
eval_set["record_id"] = [f"eval_{i:04d}" for i in range(len(eval_set))]

demo_set.to_csv(PROCESSED_DIR / "demo_set.csv", index=False)
eval_set.to_csv(PROCESSED_DIR / "eval_set.csv", index=False)

# Sanity: assert disjoint.
overlap_ids = set(demo_set["occurrenceID"]) & set(eval_set["occurrenceID"])
assert len(overlap_ids) == 0, f"DEMO/EVAL overlap detected: {overlap_ids}"

# Per-institution counts (this is what we print in the report's sampling section).
demo_inst_counts = demo_set["institutionCode"].value_counts().rename("demo_count")
eval_inst_counts = eval_set["institutionCode"].value_counts().rename("eval_count")
inst_summary = pd.concat([demo_inst_counts, eval_inst_counts], axis=1).fillna(0).astype(int)
inst_summary.to_csv(PROCESSED_DIR / "split_institution_summary.csv")

print(f"DEMO_SET: {len(demo_set)} records | EVAL_SET: {len(eval_set)} records")
print(f"Institutions in DEMO: {demo_set['institutionCode'].nunique()} | "
      f"in EVAL: {eval_set['institutionCode'].nunique()} | "
      f"intersection: {len(set(demo_set['institutionCode']) & set(eval_set['institutionCode']))}")
if demo_overlap_warning:
    print(demo_overlap_warning)
    log_event("demo_eval_split", level="warning", message=demo_overlap_warning)
else:
    print("Institution-disjoint split achieved.")
print("\nPer-institution counts:")
display(inst_summary)


In [ ]:
# ============================================================
# 5. Image download (DEMO_SET ∪ EVAL_SET)
# ============================================================
# Download images for all selected records, with retries and concurrency.
# Zenodo's image hosts occasionally throttle; we keep concurrency modest and
# back off on failure. Already-downloaded files are skipped (idempotent rerun).

def download_one(url: str, out_path: Path, timeout: int = 30, max_retries: int = 3) -> bool:
    if out_path.exists() and out_path.stat().st_size > 0:
        return True
    if not clean_str(url):
        return False
    for attempt in range(max_retries):
        try:
            r = requests.get(url, timeout=timeout,
                             headers={"User-Agent": "herbarium-scribe-v2/1.0"})
            r.raise_for_status()
            out_path.write_bytes(r.content)
            return True
        except Exception as e:
            wait = 2 ** attempt
            log_event("image_download", record_id=out_path.stem, level="warning",
                      attempt=attempt, error=repr(e), retry_wait=wait)
            time.sleep(wait)
    log_event("image_download", record_id=out_path.stem, level="error", url=url)
    return False

def download_all(df: pd.DataFrame, target_dir: Path) -> List[str]:
    tasks = []
    for _, row in df.iterrows():
        out = target_dir / f"{row['record_id']}.jpg"
        tasks.append((row["record_id"], clean_str(row.get("jpegURL", "")), out))
    paths = {rid: "" for rid, _, _ in tasks}
    # Concurrency 4: enough to hide latency, low enough not to trip Zenodo.
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(download_one, url, out): (rid, out) for rid, url, out in tasks}
        for fut in as_completed(futures):
            rid, out = futures[fut]
            if fut.result():
                paths[rid] = str(out)
    return [paths[rid] for rid, _, _ in tasks]

print("Downloading DEMO images ...")
demo_set["local_image_path"] = download_all(demo_set, IMG_DIR)
print("Downloading EVAL images ...")
eval_set["local_image_path"] = download_all(eval_set, IMG_DIR)

demo_set.to_csv(PROCESSED_DIR / "demo_set.csv", index=False)
eval_set.to_csv(PROCESSED_DIR / "eval_set.csv", index=False)

demo_ok = sum(bool(x) for x in demo_set["local_image_path"])
eval_ok = sum(bool(x) for x in eval_set["local_image_path"])
print(f"Downloaded: DEMO {demo_ok}/{len(demo_set)} | EVAL {eval_ok}/{len(eval_set)}")
if eval_ok < len(eval_set):
    print("Records without an image are still kept in the evaluation set with "
          "empty OCR text — they will score 0 on extraction. This is the honest "
          "behaviour: image-availability failures are part of pipeline performance.")


In [ ]:
# ============================================================
# 6. Build GBIF taxonomic authority subset (fixed European-flora families)
# ============================================================
# Q1(c) decision: the GBIF subset is FIXED IN ADVANCE — declared as
# EUROPEAN_FLORA_FAMILIES in the config cell — rather than derived from the
# evaluation gold. The eval set is then evaluated AGAINST this index, and
# the report prints how many gold genera/species are covered. This is more
# defensible than deriving the index from the eval gold (which would guarantee
# 100% coverage by construction and be a subtle leak).
#
# Source: GBIF /species/search API, free, no key required.
# Cache: AUTHORITY_DIR/gbif_subset.csv. Skipped if already present.

GBIF_CACHE = AUTHORITY_DIR / "gbif_subset.csv"

def fetch_gbif_family(family: str, per_family_cap: int, page_size: int = 100) -> List[Dict]:
    """Pull accepted species for a family from GBIF backbone."""
    url = "https://api.gbif.org/v1/species/search"
    rows, offset = [], 0
    while len(rows) < per_family_cap:
        params = {
            "rank": "SPECIES", "status": "ACCEPTED",
            "datasetKey": "d7dddbf4-2cf0-4f39-9b2a-bb099caae36c",  # GBIF Backbone
            "family": family, "limit": page_size, "offset": offset,
        }
        try:
            r = requests.get(url, params=params, timeout=30); r.raise_for_status()
            data = r.json()
        except Exception as e:
            log_event("gbif_fetch", level="error", family=family, error=repr(e))
            break
        results = data.get("results", [])
        if not results:
            break
        for sp in results:
            rows.append({
                "gbif_key": sp.get("key"),
                "scientific_name": sp.get("scientificName", "").strip(),
                "canonical_name": sp.get("canonicalName", "").strip(),
                "family": sp.get("family", "").strip(),
                "genus": sp.get("genus", "").strip(),
                "rank": sp.get("rank", ""),
                "status": sp.get("taxonomicStatus", ""),
            })
        offset += page_size
        if data.get("endOfRecords"):
            break
    return rows[:per_family_cap]

if GBIF_CACHE.exists():
    gbif_df = pd.read_csv(GBIF_CACHE)
    print(f"Loaded cached GBIF subset: {len(gbif_df)} accepted species")
else:
    all_rows = []
    for fam in EUROPEAN_FLORA_FAMILIES:
        rows = fetch_gbif_family(fam, GBIF_SPECIES_PER_FAMILY)
        print(f"  {fam}: {len(rows)} species")
        all_rows.extend(rows)
    gbif_df = pd.DataFrame(all_rows).drop_duplicates(subset=["canonical_name"])
    gbif_df = gbif_df[gbif_df["canonical_name"] != ""].reset_index(drop=True)
    gbif_df.to_csv(GBIF_CACHE, index=False)
    print(f"GBIF subset: {len(gbif_df)} accepted species across {gbif_df['family'].nunique()} families")

# Coverage diagnostic: how many EVAL_SET gold genera does this index cover?
# We extract the first token of each gold scientificName as a genus proxy.
gold_genera = (eval_set["scientificName"].fillna("").str.strip()
               .str.split().str[0].str.strip())
gold_genera = gold_genera[gold_genera != ""].unique()
covered_genera = set(gbif_df["genus"].str.strip().unique())
genus_coverage = sum(1 for g in gold_genera if g in covered_genera)
print(f"\nGBIF subset covers {genus_coverage} / {len(gold_genera)} EVAL gold genera "
      f"({100*genus_coverage/max(len(gold_genera),1):.1f}%).")
print("Uncovered genera (first 10):",
      [g for g in gold_genera if g not in covered_genera][:10])
log_event("gbif_subset", level="info", n_species=len(gbif_df),
          n_eval_genera=int(len(gold_genera)), n_eval_genera_covered=int(genus_coverage))


In [ ]:
# ============================================================
# 7. Build GeoNames gazetteer subset (countries + admin1 + cities15000)
# ============================================================
# Three datasets from download.geonames.org (CC-BY):
#   - countryInfo.txt       : 250 countries, ISO codes
#   - admin1CodesASCII.txt  : ~3.7k first-level admin subdivisions
#   - cities15000.zip       : ~28k cities with >15000 pop (manageable subset)
# Combined into one dataframe with a uniform "display_name" string that is
# what we embed and what we surface in the LLM prompt.

GEONAMES_CACHE = AUTHORITY_DIR / "geonames_subset.csv"
GEONAMES_BASE = "https://download.geonames.org/export/dump"

def fetch_text(url: str) -> str:
    r = requests.get(url, timeout=60); r.raise_for_status()
    return r.text

def fetch_zip_member(url: str, member_name: str) -> str:
    r = requests.get(url, timeout=120); r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        return z.read(member_name).decode("utf-8", errors="replace")

if GEONAMES_CACHE.exists():
    geonames_df = pd.read_csv(GEONAMES_CACHE)
    print(f"Loaded cached GeoNames subset: {len(geonames_df)} rows")
else:
    rows = []
    # Countries (skip comment lines starting with #)
    try:
        country_txt = fetch_text(f"{GEONAMES_BASE}/countryInfo.txt")
        country_iso_to_name = {}
        for line in country_txt.splitlines():
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split("\t")
            if len(parts) > 4:
                iso, name = parts[0], parts[4]
                country_iso_to_name[iso] = name
                rows.append({"kind": "country", "name": name, "country_code": iso,
                             "admin1_code": "", "display_name": name,
                             "lat": "", "lon": ""})
        print(f"  countries: {len(country_iso_to_name)}")
    except Exception as e:
        log_event("geonames_fetch", level="error", part="countryInfo", error=repr(e))
        country_iso_to_name = {}

    # Admin1 (e.g. "FR.A8 → Île-de-France"). Format: country.admin1_code \t name \t ascii \t geonameid
    try:
        admin1_txt = fetch_text(f"{GEONAMES_BASE}/admin1CodesASCII.txt")
        n_admin1 = 0
        for line in admin1_txt.splitlines():
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            code, name = parts[0], parts[1]
            iso = code.split(".")[0]
            admin1_code = code.split(".", 1)[1] if "." in code else ""
            country_name = country_iso_to_name.get(iso, iso)
            rows.append({"kind": "admin1", "name": name, "country_code": iso,
                         "admin1_code": admin1_code,
                         "display_name": f"{name}, {country_name}",
                         "lat": "", "lon": ""})
            n_admin1 += 1
        print(f"  admin1: {n_admin1}")
    except Exception as e:
        log_event("geonames_fetch", level="error", part="admin1", error=repr(e))

    # cities15000 — large, but only ~28k rows. Format documented at
    # https://download.geonames.org/export/dump/readme.txt
    try:
        cities_txt = fetch_zip_member(f"{GEONAMES_BASE}/cities15000.zip", "cities15000.txt")
        n_cities = 0
        for line in cities_txt.splitlines():
            parts = line.split("\t")
            if len(parts) < 11:
                continue
            name = parts[1]
            lat, lon = parts[4], parts[5]
            iso = parts[8]
            admin1_code = parts[10]
            country_name = country_iso_to_name.get(iso, iso)
            rows.append({"kind": "city", "name": name, "country_code": iso,
                         "admin1_code": admin1_code,
                         "display_name": f"{name}, {country_name}",
                         "lat": lat, "lon": lon})
            n_cities += 1
        print(f"  cities15000: {n_cities}")
    except Exception as e:
        log_event("geonames_fetch", level="error", part="cities15000", error=repr(e))

    geonames_df = pd.DataFrame(rows)
    geonames_df.to_csv(GEONAMES_CACHE, index=False)
    print(f"GeoNames subset: {len(geonames_df)} rows")

print("\nKind distribution:")
print(geonames_df["kind"].value_counts())


In [ ]:
# ============================================================
# 8. Embed authority subsets + field-guide + DEMO_SET examples
# ============================================================
# We use sentence-transformers/all-MiniLM-L6-v2:
#   - 384-dim, ~80MB on disk, runs on CPU fine for our scales
#   - tiny enough to coexist with Qwen 7B in VRAM
#   - established baseline for English bio/geo NER retrieval
#
# We build FOUR separate indices and retrieve from each at LLM time. Reason:
# concatenated retrieval over heterogeneous sources is biased toward the
# longest-text corpus. Per-source top-k keeps each grounding signal visible.

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

def encode_strings(texts: List[str], batch_size: int = 256) -> np.ndarray:
    return st_model.encode(texts, batch_size=batch_size,
                           show_progress_bar=False, normalize_embeddings=True)

# ---- Field-guide docs: short DwC field semantics ----
FIELD_GUIDE_DOCS = [
    {"doc_id": "field_catalogNumber",   "text": "catalogNumber is the barcode, catalogue number, accession number, or specimen number. It often looks like one or more uppercase letters followed by many digits, for example E00633257 or BM000123456."},
    {"doc_id": "field_institutionCode", "text": "institutionCode is the short alphabetic code for the herbarium or museum holding the specimen. Examples: BM (Natural History Museum London), E (Royal Botanic Garden Edinburgh), K (Kew), P (Muséum National d'Histoire Naturelle Paris), L (Leiden / Naturalis), BR (Meise). May be inferred from a HERBARIUM label or the prefix of the barcode."},
    {"doc_id": "field_scientificName",  "text": "scientificName is the Linnaean taxonomic name. It can include genus, species, infraspecific rank, and author string. Preserve the author string if visible (e.g. 'Quercus robur L.')."},
    {"doc_id": "field_recordedBy",      "text": "recordedBy is the collector name or expedition. May be preceded by 'Collector:', 'Coll.', 'Col.', 'leg.', 'Legit', or appear as a named expedition. Single name or list separated by '&' or ';'."},
    {"doc_id": "field_eventDate",       "text": "eventDate is the collection date. May be a full ISO-like date, a date range, or only a year. Older specimens often have only a year. Preserve as written when ambiguous."},
    {"doc_id": "field_locality",        "text": "country is the country or territory. stateProvince is the first-level admin subdivision (province, state, region, département, canton). verbatimLocality is the full written locality phrase as it appears on the label."},
    {"doc_id": "field_coordinates",     "text": "decimalLatitude and decimalLongitude are decimal-degree coordinates. If coordinates are in degrees-minutes-seconds on the label, convert. If not present in OCR, leave empty — do not guess from locality."},
    {"doc_id": "field_typeStatus",      "text": "typeStatus is a nomenclatural status: holotype, isotype, lectotype, syntype, paratype, neotype. Usually printed in red or capitals on type specimens. Leave empty if absent."},
    {"doc_id": "output_schema",         "text": "Return one valid JSON object only, with one key per target field. Each field's value must be a sub-object with keys: value (string), confidence (number 0-1), evidence_span (the OCR substring supporting the value). Do not invent values. Leave absent fields with empty value."},
]
fg_texts = [d["text"] for d in FIELD_GUIDE_DOCS]
fg_embeddings = encode_strings(fg_texts)

# ---- DEMO_SET examples: synthetic OCR-like input -> expected JSON ----
def make_synthetic_label_text(row) -> str:
    parts = []
    for prefix, field in [
        ("HERBARIUM", "institutionCode"), ("Catalogue no.", "catalogNumber"),
        ("Taxon:", "scientificName"), ("Collector:", "recordedBy"),
        ("Collected on", "eventDate"), ("Locality:", "verbatimLocality"),
        ("Country:", "country"), ("State/Province:", "stateProvince"),
        ("Type status:", "typeStatus"),
    ]:
        val = clean_str(row.get(field, ""))
        if val:
            parts.append(f"{prefix} {val}")
    if clean_str(row.get("decimalLatitude")) and clean_str(row.get("decimalLongitude")):
        parts.append(f"Lat/Long: {row['decimalLatitude']}, {row['decimalLongitude']}")
    return "\n".join(parts)

def row_to_gold_json(row) -> Dict:
    out = {}
    for field in EXTRACTION_FIELDS:
        val = clean_str(row.get(field, ""))
        out[field] = {"value": val,
                      "confidence": 1.0 if val else 0.0,
                      "evidence_span": val}
    return out

DEMO_DOCS = []
for _, row in demo_set.iterrows():
    label_text = make_synthetic_label_text(row)
    DEMO_DOCS.append({
        "doc_id": f"demo_{row['record_id']}",
        "text": ("Example label text:\n" + label_text +
                 "\nExpected JSON:\n" + json.dumps(row_to_gold_json(row), ensure_ascii=False)),
    })
demo_texts = [d["text"] for d in DEMO_DOCS]
demo_embeddings = encode_strings(demo_texts) if demo_texts else np.zeros((0, 384))

# ---- GBIF index: each row becomes a short retrieval doc ----
gbif_docs_texts = [
    f"{r['canonical_name']} (family {r['family']}, genus {r['genus']})"
    for _, r in gbif_df.iterrows()
]
gbif_embeddings = encode_strings(gbif_docs_texts, batch_size=512)

# ---- GeoNames index ----
geo_docs_texts = geonames_df["display_name"].tolist()
geo_embeddings = encode_strings(geo_docs_texts, batch_size=512)

# Cache embeddings (so reruns don't re-encode 30k+ rows)
np.save(EMBED_DIR / "fg_embeddings.npy", fg_embeddings)
np.save(EMBED_DIR / "demo_embeddings.npy", demo_embeddings)
np.save(EMBED_DIR / "gbif_embeddings.npy", gbif_embeddings)
np.save(EMBED_DIR / "geo_embeddings.npy", geo_embeddings)

print("Embeddings built:")
print(f"  field-guide: {fg_embeddings.shape}")
print(f"  demo:        {demo_embeddings.shape}")
print(f"  gbif:        {gbif_embeddings.shape}")
print(f"  geonames:    {geo_embeddings.shape}")


In [ ]:
# ============================================================
# 9. Layout detection — Hespi → PPStructure → geometric (3-tier)
# ============================================================
# Design choice (Q2): we use a herbarium-trained detector as Tier 1.
#   Tier 1: Hespi (Turnbull et al. 2024, BioScience) - YOLOv8 trained on
#           ~4k annotated MELU herbarium sheets. Returns 11 sheet-component
#           classes; we filter to "institutional_label" and "annotation_label".
#           Apache 2.0; auto-downloads weights on first use.
#           DOI: 10.1093/biosci/biaf042 ; repo: github.com/rbturnbull/hespi
#   Tier 2: PaddleOCR PPStructure (PubLayNet-trained). We get this for free
#           with the paddleocr install. Picks 'text' regions; expect lower
#           accuracy on herbarium sheets since PubLayNet is academic papers,
#           but it's a useful out-of-domain baseline + free dep.
#   Tier 3: Geometric crops (v1.4 method). Always works. Logged as fallback.
#
# Output schema per record (saved to layout_boxes.jsonl):
#   {record_id, image_path, tier, n_boxes, boxes: [{x0,y0,x1,y1,score,label}]}

LAYOUT_BOXES_PATH = PROCESSED_DIR / "layout_boxes.jsonl"

# ---------------- Tier 1: Hespi ----------------
hespi_model = None
HESPI_LABEL_CLASSES = {"institutional_label", "annotation_label", "primary_label", "secondary_label"}
if USE_HESPI:
    try:
        # Hespi exposes a Hespi() pipeline; we want only its sheet-component model.
        # The internals load YOLOv8 weights to ~/.hespi/ on first use.
        from hespi.hespi import Hespi
        hespi_model = Hespi()
        # Warm up: trigger weight download by accessing the sheet-component detector.
        _ = hespi_model.sheet_component_model
        print("Hespi loaded.")
    except Exception as e:
        log_event("hespi_load", level="error", error=repr(e))
        print("Hespi failed to load; will fall back per-image.")
        hespi_model = None

def detect_hespi(image_path: Path) -> Optional[List[Dict]]:
    if hespi_model is None:
        return None
    try:
        # The sheet_component_model is a ultralytics YOLO instance. Run it directly.
        yolo = hespi_model.sheet_component_model
        results = yolo(str(image_path), imgsz=1280, conf=0.25, verbose=False)
        boxes_out = []
        for r in results:
            names = r.names  # {id: label}
            for b in r.boxes:
                cls_id = int(b.cls.item())
                label = names.get(cls_id, str(cls_id))
                if label not in HESPI_LABEL_CLASSES:
                    continue
                x0, y0, x1, y1 = [float(v) for v in b.xyxy[0].tolist()]
                score = float(b.conf.item())
                boxes_out.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1,
                                  "score": score, "label": label})
        return boxes_out if boxes_out else None
    except Exception as e:
        log_event("hespi_detect", level="error", error=repr(e))
        return None

# ---------------- Tier 2: PPStructure layout ----------------
pp_layout = None
if USE_PPSTRUCTURE:
    try:
        from paddleocr import PPStructure
        # table=False/ocr=False: we only want layout boxes here, not table parsing.
        pp_layout = PPStructure(table=False, ocr=False, show_log=False, lang="en")
        print("PPStructure layout loaded.")
    except Exception as e:
        log_event("ppstructure_load", level="error", error=repr(e))
        pp_layout = None

def detect_ppstructure(image_path: Path) -> Optional[List[Dict]]:
    if pp_layout is None:
        return None
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        result = pp_layout(img)
        boxes_out = []
        for region in result:
            if region.get("type") not in {"text", "title", "list"}:
                continue
            bbox = region.get("bbox", [])
            if len(bbox) != 4:
                continue
            x0, y0, x1, y1 = [float(v) for v in bbox]
            boxes_out.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1,
                              "score": float(region.get("score", 0.5)),
                              "label": region.get("type", "text")})
        return boxes_out if boxes_out else None
    except Exception as e:
        log_event("ppstructure_detect", level="error", error=repr(e))
        return None

# ---------------- Tier 3: Geometric (v1.4 ratios) ----------------
def detect_geometric(image_path: Path) -> List[Dict]:
    img = cv2.imread(str(image_path))
    if img is None:
        return []
    h, w = img.shape[:2]
    specs = [
        ("lower_left",    0.00, 0.55, 0.55, 1.00),
        ("bottom_strip",  0.00, 0.78, 1.00, 1.00),
        ("lower_middle",  0.25, 0.60, 0.75, 1.00),
        ("right_strip",   0.75, 0.00, 1.00, 1.00),
        ("full",          0.00, 0.00, 1.00, 1.00),
    ]
    return [{"x0": x0*w, "y0": y0*h, "x1": x1*w, "y1": y1*h,
             "score": 0.3, "label": name} for name, x0, y0, x1, y1 in specs]

# ---------------- Driver: try Tier 1, 2, 3 in order ----------------
def detect_label_regions(image_path: Path) -> Tuple[List[Dict], str]:
    if hespi_model is not None:
        boxes = detect_hespi(image_path)
        if boxes:
            return boxes, "hespi"
    if pp_layout is not None:
        boxes = detect_ppstructure(image_path)
        if boxes:
            return boxes, "ppstructure"
    return detect_geometric(image_path), "geometric"

# Run detection over the EVAL set (DEMO not needed — DEMO uses synthetic text).
layout_rows = []
fallback_counter = {"hespi": 0, "ppstructure": 0, "geometric": 0, "no_image": 0}
all_specs = pd.concat([demo_set, eval_set], ignore_index=True)
for _, row in all_specs.iterrows():
    rid = row["record_id"]
    img_path = clean_str(row["local_image_path"])
    if not img_path or not Path(img_path).exists():
        fallback_counter["no_image"] += 1
        layout_rows.append({"record_id": rid, "image_path": "",
                            "tier": "no_image", "n_boxes": 0, "boxes": []})
        continue
    boxes, tier = detect_label_regions(Path(img_path))
    # Cap at top-3 highest-confidence boxes to keep OCR time bounded.
    boxes = sorted(boxes, key=lambda b: -b["score"])[:3]
    fallback_counter[tier] += 1
    layout_rows.append({"record_id": rid, "image_path": img_path,
                        "tier": tier, "n_boxes": len(boxes), "boxes": boxes})

# Persist as JSONL (boxes are nested, awkward in CSV).
with LAYOUT_BOXES_PATH.open("w", encoding="utf-8") as f:
    for r in layout_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("\nLayout detector tier distribution:")
for tier, count in fallback_counter.items():
    pct = 100 * count / max(len(layout_rows), 1)
    print(f"  {tier:12s} {count:4d}  ({pct:.1f}%)")
log_event("layout_summary", **fallback_counter)

# Free Hespi from VRAM before we load PaddleOCR for the next stage.
hespi_model = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 10. OCR per region — PaddleOCR (latin) primary + Tesseract fallback
# ============================================================
# Logged per crop: engine_used, region label, paddle_conf, char_count.
# Selection rule:
#   - try PaddleOCR in 'latin' script
#   - if Paddle errors OR mean line-confidence < PADDLE_LOW_CONF_THRESHOLD:
#       run Tesseract too, keep the longer non-empty output
#   - record which engine produced the final text
#
# 'latin' covers en/fr/de/nl/it/es/etc Latin-alphabet labels (covers Dillen 2019
# institutional makeup). We do NOT detect language per-record because (a) it
# adds a stage with marginal gain on 50 records, (b) latin handles the common
# case, (c) Tesseract is our second opinion when Paddle struggles.

paddle_ocr = None
if PADDLE_OK:
    try:
        from paddleocr import PaddleOCR
        paddle_ocr = PaddleOCR(use_angle_cls=True, lang=OCR_PRIMARY_LANG,
                               show_log=False)
        print(f"PaddleOCR loaded (lang={OCR_PRIMARY_LANG}).")
    except Exception as e:
        log_event("paddle_load", level="error", error=repr(e))
        paddle_ocr = None

def preprocess_for_ocr(bgr: np.ndarray) -> np.ndarray:
    """Light preprocessing — Paddle handles raw colour images well, but small
    crops benefit from upscaling. Tesseract benefits more from binarisation."""
    h, w = bgr.shape[:2]
    if max(h, w) < 800:
        bgr = cv2.resize(bgr, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    return bgr

def tesseract_ocr_one(image_bgr: np.ndarray) -> str:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    thr = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY, 35, 11)
    pil = Image.fromarray(thr)
    parts = []
    for cfg in TESS_CONFIGS:
        try:
            t = pytesseract.image_to_string(pil, lang="eng", config=cfg).strip()
            if t:
                parts.append(t)
        except Exception as e:
            log_event("tesseract", level="error", error=repr(e))
    # dedupe by exact-string
    seen, out = set(), []
    for t in parts:
        if t not in seen:
            seen.add(t); out.append(t)
    return "\n".join(out)

def paddle_ocr_one(image_bgr: np.ndarray) -> Tuple[str, float]:
    if paddle_ocr is None:
        return "", 0.0
    try:
        result = paddle_ocr.ocr(image_bgr, cls=True)
        # PaddleOCR returns a list per image. result[0] is the list of lines.
        # Each line: [bbox, (text, confidence)]
        lines = []
        if result and result[0]:
            for line in result[0]:
                if line and len(line) >= 2 and line[1]:
                    text, conf = line[1][0], float(line[1][1])
                    lines.append((text, conf))
        if not lines:
            return "", 0.0
        text = "\n".join(t for t, _ in lines)
        mean_conf = float(np.mean([c for _, c in lines]))
        return text, mean_conf
    except Exception as e:
        log_event("paddle_ocr", level="error", error=repr(e))
        return "", 0.0

def crop_image(image_bgr: np.ndarray, box: Dict) -> np.ndarray:
    h, w = image_bgr.shape[:2]
    x0 = max(0, int(box["x0"])); y0 = max(0, int(box["y0"]))
    x1 = min(w, int(box["x1"])); y1 = min(h, int(box["y1"]))
    if x1 <= x0 or y1 <= y0:
        return image_bgr
    return image_bgr[y0:y1, x0:x1]

def ocr_crop_with_fallback(image_bgr: np.ndarray) -> Dict:
    """Return {text, engine_used, paddle_conf, n_chars}."""
    prep = preprocess_for_ocr(image_bgr)
    paddle_text, paddle_conf = paddle_ocr_one(prep)
    if paddle_text and paddle_conf >= PADDLE_LOW_CONF_THRESHOLD:
        return {"text": paddle_text, "engine_used": "paddle",
                "paddle_conf": paddle_conf, "n_chars": len(paddle_text)}
    # Low-confidence path: try Tesseract; keep the better one by length
    tess_text = tesseract_ocr_one(prep)
    if not paddle_text and not tess_text:
        return {"text": "", "engine_used": "none",
                "paddle_conf": paddle_conf, "n_chars": 0}
    if len(tess_text) > len(paddle_text):
        return {"text": tess_text, "engine_used": "tesseract_fallback",
                "paddle_conf": paddle_conf, "n_chars": len(tess_text)}
    return {"text": paddle_text, "engine_used": "paddle_lowconf",
            "paddle_conf": paddle_conf, "n_chars": len(paddle_text)}

# Run OCR over every record's crops. We OCR DEMO too — we will not use DEMO's
# OCR for evaluation, but we want it available if you want to spot-check
# demo→prompt content.
ocr_region_rows = []
combined_ocr_rows = []
all_layout = {r["record_id"]: r for r in layout_rows}

records_to_ocr = pd.concat([demo_set, eval_set], ignore_index=True)
for idx, row in records_to_ocr.iterrows():
    rid = row["record_id"]
    layout = all_layout.get(rid, {"image_path": "", "boxes": []})
    img_path = layout.get("image_path", "")
    if not img_path or not Path(img_path).exists():
        combined_ocr_rows.append({"record_id": rid, "ocr_text": "",
                                  "engines_used": "", "n_crops": 0})
        continue
    img = cv2.imread(img_path)
    if img is None:
        combined_ocr_rows.append({"record_id": rid, "ocr_text": "",
                                  "engines_used": "", "n_crops": 0})
        continue
    pieces, engines = [], []
    for i, box in enumerate(layout["boxes"]):
        crop = crop_image(img, box)
        # Save the crop to disk (audit trail; the panel may want to inspect)
        crop_path = CROP_DIR / f"{rid}_box{i}_{box.get('label','x')}.png"
        cv2.imwrite(str(crop_path), crop)
        out = ocr_crop_with_fallback(crop)
        ocr_region_rows.append({
            "record_id": rid, "box_index": i, "box_label": box.get("label",""),
            "box_score": box.get("score", 0.0), "crop_path": str(crop_path),
            **out,
        })
        if out["text"]:
            pieces.append(f"[REGION={box.get('label','x')}|engine={out['engine_used']}|conf={out['paddle_conf']:.2f}]\n{out['text']}")
            engines.append(out["engine_used"])
    combined_ocr_rows.append({
        "record_id": rid,
        "ocr_text": "\n\n".join(pieces),
        "engines_used": ";".join(engines),
        "n_crops": len(layout["boxes"]),
    })
    if (idx + 1) % 10 == 0:
        print(f"  OCR'd {idx+1}/{len(records_to_ocr)}")

ocr_region_df = pd.DataFrame(ocr_region_rows)
ocr_df = pd.DataFrame(combined_ocr_rows)
ocr_region_df.to_csv(PROCESSED_DIR / "ocr_by_region.csv", index=False)
ocr_df.to_csv(PROCESSED_DIR / "ocr_combined.csv", index=False)

# Merge into eval_set / demo_set for downstream use
eval_set = eval_set.merge(ocr_df, on="record_id", how="left")
demo_set = demo_set.merge(ocr_df, on="record_id", how="left")
eval_set["ocr_text"] = eval_set["ocr_text"].fillna("")
demo_set["ocr_text"] = demo_set["ocr_text"].fillna("")
eval_set.to_csv(PROCESSED_DIR / "eval_set.csv", index=False)
demo_set.to_csv(PROCESSED_DIR / "demo_set.csv", index=False)

print("\nOCR engine distribution (across all crops):")
if len(ocr_region_df):
    eng_dist = ocr_region_df["engine_used"].value_counts()
    for k, v in eng_dist.items():
        print(f"  {k:25s} {v:4d}")
log_event("ocr_summary", n_crops=len(ocr_region_df))

# Free Paddle from VRAM before loading Qwen
paddle_ocr = None
pp_layout = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 11. OCR proxy metrics  (literal + normalised; per H1)
# ============================================================
# Why two proxies?
#   - LITERAL substring presence approximates transcription quality:
#     "did the OCR engine produce a string that contains the gold value
#     character-for-character (after lowercasing)?"
#   - NORMALISED partial-ratio approximates extraction tractability:
#     "is there enough signal in the OCR for a smart downstream extractor
#     to recover the gold value?"
#
# Both are PROXIES because Dillen 2019 ships catalogue metadata, NOT literal
# label transcriptions. We DO NOT claim CER/WER. We report two numbers
# specifically so readers can see how much of the gap is OCR vs extraction.
#
# We also bin records into OCR-quality TERTILES (low/med/high) using the
# mean normalised-presence score across that record's evaluable fields.
# Stage 13 (evaluation) stratifies metrics by these tertiles.

def norm_for_match(x: str) -> str:
    x = clean_str(x).lower()
    x = re.sub(r"[^a-z0-9.\- ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def literal_presence(value: str, text: str) -> Optional[int]:
    v = norm_for_match(value); t = norm_for_match(text)
    if not v:
        return None
    return int(v in t)

def normalised_presence(value: str, text: str) -> Optional[float]:
    v = norm_for_match(value); t = norm_for_match(text)
    if not v:
        return None
    if v in t:
        return 1.0
    return fuzz.partial_ratio(v, t) / 100.0 if t else 0.0

presence_rows = []
for _, row in eval_set.iterrows():
    txt = row["ocr_text"]
    for field in EVAL_FIELDS:
        gold_val = clean_str(row.get(field, ""))
        if not gold_val:
            continue
        presence_rows.append({
            "record_id": row["record_id"],
            "field": field,
            "gold_value": gold_val,
            "literal_present": literal_presence(gold_val, txt),
            "normalised_score": normalised_presence(gold_val, txt),
        })
ocr_presence_df = pd.DataFrame(presence_rows)
ocr_presence_df.to_csv(PROCESSED_DIR / "ocr_proxy_field_presence.csv", index=False)

# Per-field proxy summary
if len(ocr_presence_df):
    ocr_proxy_by_field = (ocr_presence_df
        .groupby("field")[["literal_present", "normalised_score"]]
        .agg(["mean", "count"]).round(3))
    print("OCR proxy by field (EVAL_SET):")
    display(ocr_proxy_by_field)

# Record-level mean normalised score → tertile
record_mean = (ocr_presence_df.groupby("record_id")["normalised_score"]
               .mean().rename("ocr_mean_presence").reset_index())
eval_set = eval_set.merge(record_mean, on="record_id", how="left")
eval_set["ocr_mean_presence"] = eval_set["ocr_mean_presence"].fillna(0.0)
# Tertile cut: qcut with duplicates='drop' for robustness when many records
# have the same score (common with low-OCR slices).
try:
    eval_set["ocr_tertile"] = pd.qcut(eval_set["ocr_mean_presence"], 3,
                                       labels=["low", "med", "high"], duplicates="drop")
except ValueError:
    eval_set["ocr_tertile"] = "med"
eval_set["ocr_tertile"] = eval_set["ocr_tertile"].astype(str).fillna("med")
eval_set.to_csv(PROCESSED_DIR / "eval_set.csv", index=False)

print("\nOCR quality tertile distribution (EVAL_SET):")
print(eval_set["ocr_tertile"].value_counts())


In [ ]:
# ============================================================
# 12. Schema validation + rule_ocr baseline extraction
# ============================================================
# Schema is unchanged from v1.4: each field is {value, confidence, evidence_span}.
# We add scientificName_verbatim / scientificName_canonical at flatten time
# (cell 16 reconciliation populates the canonical form).
#
# rule_ocr is the deterministic baseline. We KEEP it because:
#   - on stable fields (catalogNumber, coordinates, typeStatus) it often
#     ties or beats the LLM with zero compute cost
#   - it gives the interview a "show your work" anchor: every regex match
#     is auditable, no black box
# We dropped v1.4's China-specific country lookup since it was just a hack
# that flattered the demo set; replaced with a pycountry membership check.

def empty_field() -> Dict:
    return {"value": "", "confidence": 0.0, "evidence_span": ""}

def validate_record(record: Dict) -> Dict:
    """Type-coerce, range-check, accumulate warnings."""
    warnings_list = []
    for field in EXTRACTION_FIELDS:
        if field not in record or not isinstance(record[field], dict):
            record[field] = empty_field()
            warnings_list.append(f"missing_field:{field}")
        item = record[field]
        item["value"] = clean_str(item.get("value", ""))
        item["evidence_span"] = clean_str(item.get("evidence_span", ""))
        try:
            item["confidence"] = float(item.get("confidence", 0.0))
        except Exception:
            item["confidence"] = 0.0
            warnings_list.append(f"invalid_confidence:{field}")
        item["confidence"] = max(0.0, min(1.0, item["confidence"]))
    # Geo validity
    for axis, lo, hi in [("decimalLatitude", -90, 90), ("decimalLongitude", -180, 180)]:
        v = record[axis]["value"]
        if v:
            try:
                fv = float(v)
                if not (lo <= fv <= hi):
                    warnings_list.append(f"{axis}_out_of_range")
            except Exception:
                warnings_list.append(f"{axis}_not_numeric")
    record["extraction_warnings"] = warnings_list
    return record

def flatten_record(record_id: str, method: str, record: Dict) -> Dict:
    row = {"record_id": record_id, "method": method}
    for field in EXTRACTION_FIELDS:
        row[field] = record[field]["value"]
        row[f"{field}_confidence"] = record[field]["confidence"]
        row[f"{field}_evidence"] = record[field]["evidence_span"]
    row["warnings"] = ";".join(record.get("extraction_warnings", []))
    return row

# ---- rule_ocr baseline ----
def first_group(patterns, text, flags=re.IGNORECASE):
    for p in patterns:
        m = re.search(p, text, flags)
        if m:
            return m.group(1).strip(), m.group(0).strip()
    return "", ""

# pycountry-backed country lookup: scan OCR text for any country name.
# This avoids the v1.4 hardcoded "China/Australia/India/UK/France" list,
# which biased the demo set toward Forrest's China specimens.
COUNTRY_NAMES_LC = {}
for c in pycountry.countries:
    COUNTRY_NAMES_LC[c.name.lower()] = c.name
    if hasattr(c, "official_name"):
        COUNTRY_NAMES_LC[c.official_name.lower()] = c.name
    # common alternates
    if c.name == "United Kingdom":
        for alt in ("England", "Scotland", "Wales", "Northern Ireland", "Great Britain", "UK"):
            COUNTRY_NAMES_LC[alt.lower()] = "United Kingdom"

def scan_country(text: str) -> Tuple[str, str]:
    t = text.lower()
    # Longest-match first (so "United States" wins over "States")
    for name_lc in sorted(COUNTRY_NAMES_LC.keys(), key=len, reverse=True):
        if re.search(rf"\b{re.escape(name_lc)}\b", t):
            return COUNTRY_NAMES_LC[name_lc], name_lc
    return "", ""

def extract_rule_from_text(text: str) -> Dict:
    out = {f: empty_field() for f in EXTRACTION_FIELDS}

    # catalogNumber — typed prefix + digits or labelled
    v, ev = first_group([
        r"\b([A-Z]{1,4}\d{5,12})\b",
        r"(?:Catalogue|Catalog|Cat\.)\s*(?:no\.)?\s*([A-Za-z0-9_.\-]+)",
        r"(?:Barcode|barcode)\s*[:\-]?\s*([A-Za-z0-9_.\-]+)",
    ], text)
    if v:
        out["catalogNumber"] = {"value": v.strip(" ;,."), "confidence": 0.80, "evidence_span": ev}

    # institutionCode — explicit HERBARIUM or barcode prefix
    v, ev = first_group([
        r"HERBARIUM\s+([A-Z0-9_\-]{1,20})",
        r"(?:Royal Botanic Garden(?:s)?|Botanic Garden(?:s)?)\s+([A-Za-z]+)",
    ], text)
    if v:
        out["institutionCode"] = {"value": v.strip(" ;,."), "confidence": 0.60, "evidence_span": ev}
    elif out["catalogNumber"]["value"]:
        m = re.match(r"([A-Z]+)", out["catalogNumber"]["value"])
        if m:
            out["institutionCode"] = {"value": m.group(1), "confidence": 0.50,
                                       "evidence_span": out["catalogNumber"]["evidence_span"]}

    # scientificName — labelled or binomial-like pattern
    v, ev = first_group([
        r"Taxon\s*[:\-]\s*(.+)",
        r"Scientific name\s*[:\-]\s*(.+)",
        r"\b([A-Z][a-z]{2,} [a-z][a-z\-]{2,}(?:\s+\([A-Za-z.]+\))?(?:\s+[A-Z][A-Za-z.]+)?)\b",
    ], text)
    if v:
        out["scientificName"] = {"value": v.strip(" ;,.")[:200],
                                  "confidence": 0.55, "evidence_span": ev}

    # recordedBy
    v, ev = first_group([
        r"Collector\s*[:\-]\s*(.+)",
        r"Collected by\s*[:\-]\s*(.+)",
        r"Coll\.\s*([A-Za-z., \-]+)",
        r"leg\.?\s+([A-Za-z., \-]+)",
    ], text)
    if v:
        out["recordedBy"] = {"value": v.strip(" ;,.")[:200],
                              "confidence": 0.55, "evidence_span": ev}

    # eventDate
    v, ev = first_group([
        r"Collected on\s+([0-9]{4}[-/][0-9]{1,2}[-/][0-9]{1,2})",
        r"\b((?:18|19|20)\d{2}[-/]\d{1,2}[-/]\d{1,2})\b",
        r"\b((?:18|19|20)\d{2})\b",
    ], text)
    if v:
        out["eventDate"] = {"value": v.strip(" ;,."), "confidence": 0.60, "evidence_span": ev}

    # country via pycountry scan
    country, ev = scan_country(text)
    if country:
        out["country"] = {"value": country, "confidence": 0.65, "evidence_span": ev}

    # verbatimLocality — labelled
    v, ev = first_group([
        r"Locality\s*[:\-]\s*(.+)",
        r"Loc(?:ality)?\.\s*(.+)",
    ], text)
    if v:
        out["verbatimLocality"] = {"value": v.strip(" ;,.")[:300],
                                    "confidence": 0.45, "evidence_span": ev}

    # coordinates
    m = re.search(r"([-+]?\d{1,2}\.\d+)\s*[,;° ]+\s*([-+]?\d{1,3}\.\d+)", text)
    if m:
        out["decimalLatitude"]  = {"value": m.group(1), "confidence": 0.75, "evidence_span": m.group(0)}
        out["decimalLongitude"] = {"value": m.group(2), "confidence": 0.75, "evidence_span": m.group(0)}

    # typeStatus
    v, ev = first_group([r"\b(HOLOTYPE|ISOTYPE|LECTOTYPE|SYNTYPE|PARATYPE|NEOTYPE)\b"], text)
    if v:
        out["typeStatus"] = {"value": v.upper(), "confidence": 0.70, "evidence_span": ev}

    return validate_record(out)

# Quick smoke test
if len(eval_set):
    sample_rec = extract_rule_from_text(eval_set.loc[0, "ocr_text"])
    print("rule_ocr sample (eval_0000):")
    print(json.dumps({k: v for k, v in sample_rec.items() if k != "extraction_warnings"},
                     indent=2, ensure_ascii=False)[:1500])


In [ ]:
# ============================================================
# 13. Real RAG retrieval (sentence-transformer over authority + guides)
# ============================================================
# We retrieve from FOUR sources separately and concatenate top-k from each
# into the prompt. Mixing in a single ranked list is biased by source size:
# the 28k GeoNames rows would crowd out the 9 field-guide docs.
#
# Retrieval queries:
#   - the full OCR text drives the field-guide and demo retrieval
#   - capitalised tokens (likely proper nouns) drive GBIF and GeoNames retrieval
#     — this is a cheap but effective heuristic to avoid sending stop words
#     into the gazetteer index.

def top_k_cosine(query_vec: np.ndarray, doc_matrix: np.ndarray,
                 docs_text: List[str], k: int) -> List[Dict]:
    if doc_matrix.shape[0] == 0:
        return []
    # MiniLM embeddings are L2-normalised at encode time; dot product = cosine.
    scores = doc_matrix @ query_vec
    top_idx = np.argsort(scores)[::-1][:k]
    return [{"text": docs_text[i], "score": float(scores[i]), "idx": int(i)}
            for i in top_idx]

def extract_proper_noun_candidates(text: str) -> List[str]:
    """Pull capitalised n-grams that look like proper nouns. Used as query for
    GBIF and GeoNames retrieval. Drops common all-caps shouty words."""
    NOISE = {"HERBARIUM", "MUSEUM", "NATURAL", "HISTORY", "TYPE",
             "HOLOTYPE", "ISOTYPE", "LECTOTYPE", "DET", "LEG",
             "COLLECTOR", "COLL", "LOCALITY"}
    candidates = re.findall(r"\b[A-Z][a-zA-Z\-]+(?:\s+[A-Z][a-zA-Z\-]+){0,3}\b", text)
    return [c for c in candidates if c.upper() not in NOISE][:8]

def retrieve_context_v2(ocr_text: str, k_fg=3, k_demo=2, k_gbif=4, k_geo=4) -> Dict:
    q_main = ocr_text if clean_str(ocr_text) else "empty OCR herbarium specimen"
    q_main_vec = encode_strings([q_main])[0]
    fg_hits   = top_k_cosine(q_main_vec, fg_embeddings, fg_texts, k_fg)
    demo_hits = top_k_cosine(q_main_vec, demo_embeddings, demo_texts, k_demo) if demo_embeddings.shape[0] else []

    proper_nouns = extract_proper_noun_candidates(ocr_text)
    q_pn = " ".join(proper_nouns) if proper_nouns else q_main
    q_pn_vec = encode_strings([q_pn])[0]
    gbif_hits = top_k_cosine(q_pn_vec, gbif_embeddings, gbif_docs_texts, k_gbif)
    geo_hits  = top_k_cosine(q_pn_vec, geo_embeddings, geo_docs_texts, k_geo)

    return {"field_guide": fg_hits, "demo": demo_hits,
            "gbif": gbif_hits, "geonames": geo_hits,
            "proper_nouns": proper_nouns}

def format_retrieval_for_prompt(retrieved: Dict) -> str:
    parts = []
    if retrieved["field_guide"]:
        parts.append("FIELD GUIDE (Darwin Core field semantics):")
        for h in retrieved["field_guide"]:
            parts.append(f"- (sim={h['score']:.2f}) {h['text']}")
    if retrieved["demo"]:
        parts.append("\nEXAMPLES (from DEMO_SET, disjoint from EVAL_SET):")
        for h in retrieved["demo"]:
            parts.append(f"- (sim={h['score']:.2f}) {h['text']}")
    if retrieved["gbif"]:
        parts.append("\nGBIF candidate taxa (accepted names from the indexed European-flora subset):")
        for h in retrieved["gbif"]:
            parts.append(f"- (sim={h['score']:.2f}) {h['text']}")
    if retrieved["geonames"]:
        parts.append("\nGEONAMES candidate places (countries / admin1 / cities ≥15000 pop):")
        for h in retrieved["geonames"]:
            parts.append(f"- (sim={h['score']:.2f}) {h['text']}")
    return "\n".join(parts)

# Smoke test
if len(eval_set):
    sample_retrieval = retrieve_context_v2(eval_set.loc[0, "ocr_text"])
    print("Sample retrieval for eval_0000:")
    print(format_retrieval_for_prompt(sample_retrieval)[:2500])


In [ ]:
# ============================================================
# 14. LLM backend — call_llm(messages) -> str
# ============================================================
# Two backends behind one signature. Switch with LLM_BACKEND in cell 2.
#   - "qwen"      : Qwen2.5-7B-Instruct, 4-bit nf4, deterministic decode
#   - "anthropic" : Anthropic Messages API, model = ANTHROPIC_MODEL
#                   Requires env var ANTHROPIC_API_KEY. Never hardcoded.
#
# Why this abstraction matters for the demo: Colab's free T4 occasionally
# OOMs mid-run. If that happens, set LLM_BACKEND='anthropic' and rerun from
# the JSONL checkpoint — no other code changes.

qwen_tokenizer = None
qwen_model = None

def load_qwen():
    """Load Qwen 7B in 4-bit. Idempotent."""
    global qwen_tokenizer, qwen_model
    if qwen_model is not None:
        return
    qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
    kwargs = {"device_map": "auto", "trust_remote_code": True}
    if QWEN_USE_4BIT and torch.cuda.is_available():
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    else:
        kwargs["torch_dtype"] = torch.float16 if torch.cuda.is_available() else torch.float32
    qwen_model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL_ID, **kwargs)
    qwen_model.eval()

def call_qwen(messages: List[Dict], max_new_tokens: int = 800) -> str:
    if qwen_model is None:
        load_qwen()
    prompt = qwen_tokenizer.apply_chat_template(messages, tokenize=False,
                                                 add_generation_prompt=True)
    inputs = qwen_tokenizer([prompt], return_tensors="pt",
                             truncation=True, max_length=6144).to(qwen_model.device)
    with torch.no_grad():
        out = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(gen, skip_special_tokens=True)

def call_anthropic(messages: List[Dict], max_new_tokens: int = 800) -> str:
    import anthropic
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        raise RuntimeError("LLM_BACKEND='anthropic' but ANTHROPIC_API_KEY not set in env.")
    client = anthropic.Anthropic(api_key=key)
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    user_msgs = [{"role": m["role"], "content": m["content"]}
                 for m in messages if m["role"] != "system"]
    resp = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=max_new_tokens,
        system=system,
        messages=user_msgs,
    )
    # Concatenate all text blocks (Anthropic content is a list of blocks).
    parts = []
    for block in resp.content:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
    return "".join(parts)

def call_llm(messages: List[Dict], max_new_tokens: int = 800) -> str:
    if LLM_BACKEND == "qwen":
        return call_qwen(messages, max_new_tokens)
    elif LLM_BACKEND == "anthropic":
        return call_anthropic(messages, max_new_tokens)
    else:
        raise ValueError(f"Unknown LLM_BACKEND: {LLM_BACKEND!r}")

# Eager-load the backend so we hit any failure now, not in the middle of the
# extraction loop.
print(f"LLM backend: {LLM_BACKEND}")
if LLM_BACKEND == "qwen":
    print("Loading Qwen ... (~3-5 min in 4-bit)")
    t0 = time.time()
    load_qwen()
    print(f"Qwen loaded in {time.time()-t0:.1f}s")
else:
    # Cheap probe of the API
    _ = call_anthropic([{"role": "user", "content": "ping"}], max_new_tokens=10)
    print(f"Anthropic backend OK (model={ANTHROPIC_MODEL})")


In [ ]:
# ============================================================
# 15. Run all three extraction methods on EVAL_SET
# ============================================================
# Three flat outputs to a single CSV (one row per record × method),
# plus a JSONL with the raw LLM output (audit trail).
# All outputs are checkpointed: if extraction crashes at record k, restart
# from k by reading the JSONL and skipping records already present.

def extract_json_from_text(text: str) -> Optional[Dict]:
    """Robust JSON parsing of LLM output. Handles ```json fences and prefixed prose."""
    text = clean_str(text)
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except Exception:
        pass
    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            return None
    return None

def normalise_llm_json(obj: Any) -> Dict:
    rec = {}
    for field in EXTRACTION_FIELDS:
        item = obj.get(field, {}) if isinstance(obj, dict) else {}
        if isinstance(item, dict):
            value = clean_str(item.get("value", ""))
            conf = item.get("confidence", 0.0)
            ev = clean_str(item.get("evidence_span", ""))
        else:
            value = clean_str(item); conf = 0.5 if value else 0.0; ev = value
        try:
            conf = float(conf)
        except Exception:
            conf = 0.0
        rec[field] = {"value": value, "confidence": max(0.0, min(1.0, conf)), "evidence_span": ev}
    return validate_record(rec)

SYSTEM_MSG = (
    "You extract structured biodiversity specimen data from OCR text of "
    "herbarium specimen labels. Return strict JSON only. Do not invent values. "
    "If evidence is not visible in the OCR text, leave the value empty."
)

USER_TEMPLATE = """OCR text from one specimen sheet:
---
{ocr}
---
{retrieved_block}
Extract these Darwin Core fields:
{fields}

Return ONE valid JSON object. Each field must be an object with keys:
- value: string  (empty string if not present in OCR)
- confidence: number from 0 to 1
- evidence_span: the exact OCR substring that supports the value (or empty)

Rules:
- No markdown, no preamble, JSON only.
- Do not invent. If you are unsure, leave value empty.
- Use the retrieved candidate taxa and place names ONLY as hints when the OCR
  text is ambiguous; never substitute them for clearly-different OCR content.
- For scientificName, prefer the full binomial as written on the label; the
  canonical form will be resolved downstream by GBIF.
"""

def build_messages(ocr_text: str, retrieved: Optional[Dict] = None) -> List[Dict]:
    retrieved_block = ""
    if retrieved:
        retrieved_block = "\nRetrieved context (use as hints, not substitutions):\n" + \
                          format_retrieval_for_prompt(retrieved) + "\n"
    user = USER_TEMPLATE.format(ocr=ocr_text or "(empty)",
                                 retrieved_block=retrieved_block,
                                 fields=EXTRACTION_FIELDS)
    return [{"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": user}]

def llm_extract(ocr_text: str, use_rag: bool) -> Tuple[Dict, str, Optional[Dict]]:
    retrieved = retrieve_context_v2(ocr_text) if use_rag else None
    messages = build_messages(ocr_text, retrieved)
    raw = call_llm(messages, max_new_tokens=800)
    parsed = extract_json_from_text(raw)
    if parsed is None:
        empty = {f: empty_field() for f in EXTRACTION_FIELDS}
        empty = validate_record(empty)
        empty["extraction_warnings"].append("llm_json_parse_failed")
        return empty, raw, retrieved
    return normalise_llm_json(parsed), raw, retrieved

# ---- Checkpointed extraction loop ----
EXTRACT_JSONL = PROCESSED_DIR / "extractions.jsonl"
done_keys = set()
if EXTRACT_JSONL.exists():
    with EXTRACT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                done_keys.add((obj["record_id"], obj["method"]))
            except Exception:
                pass
    print(f"Resuming: {len(done_keys)} (record_id, method) combos already done.")

flat_rows: List[Dict] = []
retrieval_rows: List[Dict] = []
methods = ["rule_ocr", "qwen_no_rag", "qwen_rag_v2"]

with EXTRACT_JSONL.open("a", encoding="utf-8") as fout:
    for i, row in eval_set.iterrows():
        rid = row["record_id"]
        ocr_text = clean_str(row.get("ocr_text", ""))
        for method in methods:
            if (rid, method) in done_keys:
                continue
            t0 = time.time()
            if method == "rule_ocr":
                rec = extract_rule_from_text(ocr_text)
                raw = ""
                retrieved = None
            elif method == "qwen_no_rag":
                rec, raw, retrieved = llm_extract(ocr_text, use_rag=False)
            else:  # qwen_rag_v2
                rec, raw, retrieved = llm_extract(ocr_text, use_rag=True)
            flat = flatten_record(rid, method, rec)
            flat["elapsed_s"] = round(time.time() - t0, 2)
            flat_rows.append(flat)
            nested = {"record_id": rid, "method": method,
                      "ocr_text": ocr_text,
                      "raw_model_output": raw,
                      "extraction": rec,
                      "elapsed_s": flat["elapsed_s"]}
            if retrieved:
                nested["retrieved_doc_summary"] = {
                    "field_guide_top": [h["text"][:120] for h in retrieved["field_guide"]],
                    "demo_top":         [h["text"][:120] for h in retrieved["demo"]],
                    "gbif_top":         [h["text"] for h in retrieved["gbif"]],
                    "geonames_top":     [h["text"] for h in retrieved["geonames"]],
                    "proper_nouns":     retrieved["proper_nouns"],
                }
                for src in ("field_guide", "demo", "gbif", "geonames"):
                    for rank, h in enumerate(retrieved[src], start=1):
                        retrieval_rows.append({"record_id": rid, "method": method,
                                                "source": src, "rank": rank,
                                                "score": h["score"],
                                                "text": h["text"][:200]})
            fout.write(json.dumps(nested, ensure_ascii=False) + "\n")
            fout.flush()
        if (i + 1) % 5 == 0:
            print(f"  done {i+1}/{len(eval_set)}")

# Reload everything from JSONL (covers resumed runs that mixed old + new rows)
nested_records = []
with EXTRACT_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        try:
            nested_records.append(json.loads(line))
        except Exception:
            pass

# Build the flat dataframe from JSONL (single source of truth after resume).
flat_all = []
for n in nested_records:
    flat = flatten_record(n["record_id"], n["method"], n["extraction"])
    flat["elapsed_s"] = n.get("elapsed_s", 0.0)
    flat_all.append(flat)
pred_all = pd.DataFrame(flat_all)
pred_all.to_csv(PROCESSED_DIR / "extractions_flat.csv", index=False)

if retrieval_rows:
    retrieval_df = pd.DataFrame(retrieval_rows)
    retrieval_df.to_csv(PROCESSED_DIR / "rag_retrieval_log.csv", index=False)

print(f"\nExtraction complete: {len(pred_all)} rows across "
      f"{pred_all['record_id'].nunique()} records × {pred_all['method'].nunique()} methods")
print("\nMean elapsed by method (seconds):")
print(pred_all.groupby("method")["elapsed_s"].mean().round(2))


In [ ]:
# ============================================================
# 16. Reconciliation — GBIF /species/match + country/admin1 grounding
# ============================================================
# Design (Q5 + Q6):
#   - scientificName: GBIF /species/match (verbose) is primary. It returns a
#     typed matchType (EXACT/FUZZY/HIGHERRANK/NONE) plus the accepted-name
#     canonical and a synonym flag. We store BOTH verbatim and canonical;
#     we NEVER silently overwrite the LLM output. Synonyms get a flag.
#   - When GBIF returns NONE, we fall back to cosine over the local GBIF
#     subset using the three thresholds in the config cell.
#   - country: pycountry exact match primary; GeoNames cosine fallback.
#   - stateProvince: GeoNames admin1 cosine, optionally filtered by country.
#
# An on-disk cache (gbif_match_cache.json) holds API responses so reruns
# don't repeat HTTP calls.

GBIF_CACHE_PATH = AUTHORITY_DIR / "gbif_match_cache.json"
gbif_match_cache: Dict[str, Dict] = {}
if GBIF_CACHE_PATH.exists():
    try:
        gbif_match_cache = json.loads(GBIF_CACHE_PATH.read_text(encoding="utf-8"))
    except Exception:
        gbif_match_cache = {}

def save_gbif_cache():
    GBIF_CACHE_PATH.write_text(json.dumps(gbif_match_cache, ensure_ascii=False),
                                encoding="utf-8")

def gbif_species_match(name: str) -> Dict:
    """Call GBIF /species/match. Cached. Returns the raw API response (dict)."""
    key = name.strip().lower()
    if key in gbif_match_cache:
        return gbif_match_cache[key]
    try:
        r = requests.get("https://api.gbif.org/v1/species/match",
                         params={"name": name, "verbose": "true"}, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        log_event("gbif_match", level="error", name=name, error=repr(e))
        data = {"matchType": "ERROR", "error": repr(e)}
    gbif_match_cache[key] = data
    return data

def cosine_fallback_gbif(name: str) -> Tuple[str, float]:
    """Best cosine match in the local GBIF subset. Returns (canonical, score)."""
    if not name.strip() or gbif_embeddings.shape[0] == 0:
        return "", 0.0
    q = encode_strings([name])[0]
    scores = gbif_embeddings @ q
    j = int(np.argmax(scores))
    return gbif_df.iloc[j]["canonical_name"], float(scores[j])

def reconcile_scientific_name(verbatim: str) -> Dict:
    """
    Returns a dict with verbatim, canonical, match_type, match_confidence,
    is_synonym, gbif_usage_key, warnings.
    """
    result = {
        "verbatim": verbatim, "canonical": "", "match_type": "EMPTY",
        "match_confidence": 0.0, "is_synonym": False,
        "gbif_usage_key": None, "warnings": [],
    }
    if not clean_str(verbatim):
        return result

    # Primary: GBIF /species/match
    gbif_resp = gbif_species_match(verbatim)
    mt = gbif_resp.get("matchType", "NONE")
    if mt in {"EXACT", "FUZZY"} and gbif_resp.get("usageKey"):
        canonical = gbif_resp.get("canonicalName", "") or ""
        is_synonym = bool(gbif_resp.get("synonym", False))
        result.update({
            "canonical": canonical,
            "match_type": mt,
            "match_confidence": gbif_resp.get("confidence", 0) / 100.0,
            "is_synonym": is_synonym,
            "gbif_usage_key": gbif_resp.get("acceptedUsageKey") or gbif_resp.get("usageKey"),
        })
        if is_synonym:
            # Q6: never silently overwrite. Flag and keep both.
            result["warnings"].append("synonym_match")
        return result

    if mt == "HIGHERRANK":
        result.update({"match_type": "HIGHERRANK",
                       "match_confidence": gbif_resp.get("confidence", 0) / 100.0,
                       "canonical": gbif_resp.get("canonicalName", "")})
        result["warnings"].append("only_higherrank_match")
        return result

    # Fallback: cosine over local subset (used when GBIF is unreachable or NONE).
    cand, sim = cosine_fallback_gbif(verbatim)
    if sim >= COS_AUTO_REPLACE:
        result.update({"canonical": cand, "match_type": "COSINE_HIGH",
                       "match_confidence": sim})
        result["warnings"].append("fallback_cosine_auto")
    elif sim >= COS_FLAG_REVIEW:
        result.update({"canonical": cand, "match_type": "COSINE_FLAG",
                       "match_confidence": sim})
        result["warnings"].append("fallback_cosine_flag_for_review")
    else:
        result.update({"match_type": "NONE", "match_confidence": sim})
        result["warnings"].append("no_gbif_or_cosine_match")
    return result

# Country reconciliation — pycountry primary, GeoNames cosine secondary.
def reconcile_country(verbatim: str) -> Dict:
    out = {"verbatim": verbatim, "canonical": "", "method": "EMPTY",
           "confidence": 0.0, "warnings": []}
    if not clean_str(verbatim):
        return out
    v = verbatim.strip()
    # Exact name or ISO-2/ISO-3 lookup
    try:
        c = pycountry.countries.lookup(v)
        out.update({"canonical": c.name, "method": "pycountry_exact",
                    "confidence": 1.0})
        return out
    except LookupError:
        pass
    # Substring scan (e.g. "Yunnan, West China" → China)
    name, _ = scan_country(v)
    if name:
        out.update({"canonical": name, "method": "pycountry_substring",
                    "confidence": 0.9})
        return out
    # Cosine over GeoNames countries
    country_mask = geonames_df["kind"] == "country"
    country_idx = np.where(country_mask.values)[0]
    if len(country_idx):
        q = encode_strings([v])[0]
        country_emb = geo_embeddings[country_idx]
        scores = country_emb @ q
        j = int(np.argmax(scores))
        sim = float(scores[j])
        cand = geonames_df.iloc[country_idx[j]]["name"]
        if sim >= COS_AUTO_REPLACE:
            out.update({"canonical": cand, "method": "geonames_cosine",
                        "confidence": sim})
        elif sim >= COS_FLAG_REVIEW:
            out.update({"canonical": cand, "method": "geonames_cosine_flag",
                        "confidence": sim, "warnings": ["country_flag_for_review"]})
        else:
            out.update({"canonical": "", "method": "no_match",
                        "confidence": sim, "warnings": ["country_no_match"]})
    return out

def reconcile_state(verbatim: str, country_iso_or_name: str = "") -> Dict:
    """Cosine over GeoNames admin1, optionally filtered by country."""
    out = {"verbatim": verbatim, "canonical": "", "method": "EMPTY",
           "confidence": 0.0, "warnings": []}
    if not clean_str(verbatim):
        return out
    # Build mask
    mask = geonames_df["kind"] == "admin1"
    if country_iso_or_name:
        iso = ""
        try:
            iso = pycountry.countries.lookup(country_iso_or_name).alpha_2
        except LookupError:
            pass
        if iso:
            mask = mask & (geonames_df["country_code"] == iso)
    idx = np.where(mask.values)[0]
    if not len(idx):
        out["warnings"].append("admin1_no_index")
        return out
    q = encode_strings([verbatim])[0]
    sub_emb = geo_embeddings[idx]
    scores = sub_emb @ q
    j = int(np.argmax(scores))
    sim = float(scores[j])
    cand = geonames_df.iloc[idx[j]]["name"]
    if sim >= COS_AUTO_REPLACE:
        out.update({"canonical": cand, "method": "geonames_admin1_cosine",
                    "confidence": sim})
    elif sim >= COS_FLAG_REVIEW:
        out.update({"canonical": cand, "method": "geonames_admin1_cosine_flag",
                    "confidence": sim, "warnings": ["state_flag_for_review"]})
    else:
        out.update({"method": "no_match", "confidence": sim,
                    "warnings": ["state_no_match"]})
    return out

# Apply reconciliation to every (record, method) row.
recon_rows = []
for _, r in pred_all.iterrows():
    sn = reconcile_scientific_name(r.get("scientificName", ""))
    co = reconcile_country(r.get("country", ""))
    # Use the reconciled country to constrain the admin1 search
    sp = reconcile_state(r.get("stateProvince", ""),
                         country_iso_or_name=co.get("canonical", ""))
    recon_rows.append({
        "record_id": r["record_id"],
        "method":    r["method"],
        "scientificName_verbatim":   sn["verbatim"],
        "scientificName_canonical":  sn["canonical"],
        "scientificName_match_type": sn["match_type"],
        "scientificName_match_conf": sn["match_confidence"],
        "scientificName_is_synonym": sn["is_synonym"],
        "scientificName_gbif_key":   sn["gbif_usage_key"],
        "scientificName_warnings":   ";".join(sn["warnings"]),
        "country_verbatim":   co["verbatim"],
        "country_canonical":  co["canonical"],
        "country_method":     co["method"],
        "country_conf":       co["confidence"],
        "country_warnings":   ";".join(co["warnings"]),
        "stateProvince_verbatim":  sp["verbatim"],
        "stateProvince_canonical": sp["canonical"],
        "stateProvince_method":    sp["method"],
        "stateProvince_conf":      sp["confidence"],
        "stateProvince_warnings":  ";".join(sp["warnings"]),
    })
recon_df = pd.DataFrame(recon_rows)
recon_df.to_csv(PROCESSED_DIR / "reconciliation.csv", index=False)
save_gbif_cache()

# Merge canonical forms into the flat prediction table so downstream
# evaluation can score against EITHER verbatim OR canonical.
pred_all_recon = pred_all.merge(recon_df, on=["record_id", "method"], how="left")
pred_all_recon.to_csv(PROCESSED_DIR / "extractions_flat_reconciled.csv", index=False)

# Reconciliation distribution diagnostics
print("\nGBIF /species/match types across all (record × method) extractions:")
print(recon_df["scientificName_match_type"].value_counts())
print("\nCountry reconciliation methods:")
print(recon_df["country_method"].value_counts())
print("\nCosine similarity distribution (scientificName, where match was cosine):")
cos_mask = recon_df["scientificName_match_type"].str.startswith("COSINE")
if cos_mask.any():
    print(recon_df.loc[cos_mask, "scientificName_match_conf"].describe().round(3))


In [ ]:
# ============================================================
# 17. Evaluation — per-field × per-method, stratified by OCR tertile,
#     with bootstrap 95% CIs on headline numbers
# ============================================================
# Two evaluation "modes" for scientificName:
#   - VERBATIM mode: compare LLM/rule output to gold scientificName (matches v1.4)
#   - CANONICAL mode: compare GBIF-reconciled canonical to gold (post-reconciliation)
# Both reported. Difference between them = lift from reconciliation.

def normalize_eval(x: str) -> str:
    x = clean_str(x).lower()
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^\w\s.\-]", "", x)
    return x.strip()

def exact_match(p: str, g: str) -> int:
    return int(normalize_eval(p) == normalize_eval(g))

def token_f1(p: str, g: str) -> float:
    pt, gt = normalize_eval(p).split(), normalize_eval(g).split()
    if not pt and not gt: return 1.0
    if not pt or not gt:  return 0.0
    pc, gc = {}, {}
    for t in pt: pc[t] = pc.get(t, 0) + 1
    for t in gt: gc[t] = gc.get(t, 0) + 1
    common = sum(min(pc.get(t, 0), gc.get(t, 0)) for t in pc)
    if common == 0: return 0.0
    prec, rec = common / len(pt), common / len(gt)
    return 2 * prec * rec / (prec + rec)

def per_field_eval(pred_df: pd.DataFrame, gold_df: pd.DataFrame,
                    fields: List[str],
                    pred_col_resolver=None) -> pd.DataFrame:
    """Compute coverage, EM, F1 per (method, field). pred_col_resolver lets us
    swap in `scientificName_canonical` for the canonical evaluation."""
    rows = []
    for method, sub in pred_df.groupby("method"):
        merged = sub.merge(gold_df[["record_id"] + fields], on="record_id",
                           suffixes=("_pred", "_gold"))
        for field in fields:
            pcol = field + "_pred"
            gcol = field + "_gold"
            if pred_col_resolver:
                # Use a different prediction column for this field
                custom = pred_col_resolver(field)
                if custom and custom in sub.columns:
                    merged_custom = sub[["record_id", custom]].merge(
                        gold_df[["record_id", field]], on="record_id")
                    pcol = custom
                    gcol = field
                    merged_field = merged_custom
                else:
                    merged_field = merged
            else:
                merged_field = merged
            evaluable = merged_field[gcol].apply(clean_str) != ""
            sub_eval = merged_field.loc[evaluable].copy()
            if len(sub_eval) == 0:
                rows.append({"method": method, "field": field, "n_evaluable": 0,
                             "prediction_coverage": np.nan,
                             "exact_match": np.nan, "token_f1": np.nan})
                continue
            cov = (sub_eval[pcol].apply(clean_str) != "").mean()
            ems = [exact_match(p, g) for p, g in zip(sub_eval[pcol], sub_eval[gcol])]
            f1s = [token_f1(p, g)    for p, g in zip(sub_eval[pcol], sub_eval[gcol])]
            rows.append({"method": method, "field": field,
                         "n_evaluable": int(len(sub_eval)),
                         "prediction_coverage": float(cov),
                         "exact_match": float(np.mean(ems)),
                         "token_f1": float(np.mean(f1s))})
    return pd.DataFrame(rows)

# ---- 1. Verbatim-field evaluation (baseline view) ----
eval_verbatim = per_field_eval(pred_all_recon, eval_set, EVAL_FIELDS)
eval_verbatim["mode"] = "verbatim"
eval_verbatim.to_csv(PROCESSED_DIR / "eval_verbatim.csv", index=False)

# ---- 2. Canonical evaluation: scientificName & country swapped for reconciled form ----
def canonical_resolver(field):
    return {
        "scientificName": "scientificName_canonical",
        "country":        "country_canonical",
        "stateProvince":  "stateProvince_canonical",
    }.get(field)

eval_canonical = per_field_eval(pred_all_recon, eval_set, EVAL_FIELDS,
                                pred_col_resolver=canonical_resolver)
eval_canonical["mode"] = "canonical"
eval_canonical.to_csv(PROCESSED_DIR / "eval_canonical.csv", index=False)

# ---- 3. Stratified by OCR tertile ----
strat_rows = []
for tertile in ["low", "med", "high"]:
    rids = set(eval_set.loc[eval_set["ocr_tertile"] == tertile, "record_id"])
    if not rids:
        continue
    pred_sub = pred_all_recon[pred_all_recon["record_id"].isin(rids)]
    gold_sub = eval_set[eval_set["record_id"].isin(rids)]
    if pred_sub.empty:
        continue
    sub_eval = per_field_eval(pred_sub, gold_sub, EVAL_FIELDS)
    sub_eval["ocr_tertile"] = tertile
    sub_eval["n_records_in_tertile"] = len(rids)
    strat_rows.append(sub_eval)

if strat_rows:
    eval_stratified = pd.concat(strat_rows, ignore_index=True)
else:
    eval_stratified = pd.DataFrame()
eval_stratified.to_csv(PROCESSED_DIR / "eval_stratified_by_ocr_tertile.csv", index=False)

# ---- 4. Method summary with bootstrap 95% CI on overall token_f1 ----
def bootstrap_ci_f1(pred_df: pd.DataFrame, gold_df: pd.DataFrame,
                    fields: List[str], n_boot: int = 200,
                    seed: int = 42) -> pd.DataFrame:
    """Bootstrap over RECORDS (correct unit), not field-record pairs."""
    rng = np.random.RandomState(seed)
    out = []
    for method, sub in pred_df.groupby("method"):
        merged = sub.merge(gold_df[["record_id"] + fields], on="record_id",
                            suffixes=("_pred", "_gold"))
        records = merged["record_id"].unique()
        if len(records) == 0:
            continue
        # Per-record mean F1 across fields (with non-empty gold).
        per_record_f1 = []
        for rid in records:
            r = merged[merged["record_id"] == rid].iloc[0]
            f1s = []
            for f in fields:
                gv = clean_str(r[f + "_gold"])
                if not gv:
                    continue
                pv = clean_str(r[f + "_pred"])
                f1s.append(token_f1(pv, gv))
            if f1s:
                per_record_f1.append(np.mean(f1s))
        per_record_f1 = np.array(per_record_f1)
        if len(per_record_f1) == 0:
            continue
        mean_f1 = float(per_record_f1.mean())
        boot_means = []
        for _ in range(n_boot):
            sample = rng.choice(per_record_f1, size=len(per_record_f1), replace=True)
            boot_means.append(sample.mean())
        lo, hi = np.percentile(boot_means, [2.5, 97.5])
        out.append({"method": method,
                    "mean_token_f1": mean_f1,
                    "ci_lo": float(lo), "ci_hi": float(hi),
                    "n_records": len(per_record_f1)})
    return pd.DataFrame(out)

method_summary = bootstrap_ci_f1(pred_all_recon, eval_set, EVAL_FIELDS, n_boot=500)
method_summary.to_csv(PROCESSED_DIR / "method_summary_with_ci.csv", index=False)

print("Per-method summary (verbatim, bootstrap 95% CI):")
display(method_summary.round(3))

print("\nVerbatim per-field metrics:")
display(eval_verbatim.round(3))

print("\nCanonical (after GBIF/GeoNames reconciliation) per-field metrics:")
display(eval_canonical.round(3))

print("\nStratified by OCR-quality tertile:")
display(eval_stratified.round(3))


In [ ]:
# ============================================================
# 18. Graph export with light collector entity resolution
# ============================================================
# Source method for graph: qwen_rag_v2 (preferred), fall back to rule_ocr if
# qwen wasn't run. We add a small entity-resolution step on `recordedBy`:
# names with cosine similarity ≥ 0.92 are merged onto the canonical form
# (chosen as the longest variant — usually "George Forrest" wins over "G. Forrest").
# scientificName / country use the reconciled canonical forms from cell 16.

GRAPH_METHOD = "qwen_rag_v2" if "qwen_rag_v2" in pred_all_recon["method"].unique() else "rule_ocr"
g_src = pred_all_recon[pred_all_recon["method"] == GRAPH_METHOD].copy()

# Collector entity resolution
all_collectors = sorted({clean_str(v) for v in g_src["recordedBy"] if clean_str(v)})
collector_canonical = {c: c for c in all_collectors}
if len(all_collectors) >= 2:
    coll_emb = encode_strings(all_collectors)
    sims = coll_emb @ coll_emb.T
    # Greedy clustering: for each name, merge into the longest name within sim >= 0.92
    for i, name_i in enumerate(all_collectors):
        partners = [j for j in range(len(all_collectors)) if j != i and sims[i, j] >= 0.92]
        if partners:
            cluster = [name_i] + [all_collectors[j] for j in partners]
            canonical = max(cluster, key=len)  # longest = most informative
            collector_canonical[name_i] = canonical

g_src["recordedBy_canonical"] = g_src["recordedBy"].apply(
    lambda v: collector_canonical.get(clean_str(v), clean_str(v)))

nodes, edges = [], []
def add_node(node_id, label, name, extra=None):
    if not clean_str(name): return
    row = {"id": node_id, "label": label, "name": clean_str(name)}
    if extra: row.update(extra)
    nodes.append(row)

for _, r in g_src.iterrows():
    spec_name = clean_str(r.get("catalogNumber")) or clean_str(r.get("record_id"))
    spec_id = f"specimen:{spec_name}"
    add_node(spec_id, "Specimen", spec_name,
             extra={"institutionCode": clean_str(r.get("institutionCode"))})

    # Taxon — prefer canonical, else verbatim
    taxon_value = clean_str(r.get("scientificName_canonical")) or clean_str(r.get("scientificName"))
    if taxon_value:
        t_id = f"taxon:{taxon_value}"
        add_node(t_id, "Taxon", taxon_value,
                 extra={"gbif_key": r.get("scientificName_gbif_key"),
                         "match_type": r.get("scientificName_match_type"),
                         "is_synonym": bool(r.get("scientificName_is_synonym"))})
        edges.append({"source": spec_id, "target": t_id, "relationship": "HAS_TAXON"})

    # Collector — canonical
    coll = clean_str(r.get("recordedBy_canonical"))
    if coll:
        c_id = f"collector:{coll}"
        add_node(c_id, "Collector", coll)
        edges.append({"source": spec_id, "target": c_id, "relationship": "COLLECTED_BY"})

    # Place — country canonical
    country = clean_str(r.get("country_canonical")) or clean_str(r.get("country"))
    if country:
        p_id = f"place:{country}"
        add_node(p_id, "Place", country)
        edges.append({"source": spec_id, "target": p_id, "relationship": "COLLECTED_IN"})

    # Institution
    inst = clean_str(r.get("institutionCode"))
    if inst:
        i_id = f"institution:{inst}"
        add_node(i_id, "Institution", inst)
        edges.append({"source": spec_id, "target": i_id, "relationship": "HELD_BY"})

nodes_df = pd.DataFrame(nodes).drop_duplicates(subset=["id"])
edges_df = pd.DataFrame(edges).drop_duplicates()
nodes_df.to_csv(PROCESSED_DIR / "graph_nodes.csv", index=False)
edges_df.to_csv(PROCESSED_DIR / "graph_edges.csv", index=False)

# Quick sanity: how much did entity resolution collapse the collector node count?
n_collectors_before = len(all_collectors)
n_collectors_after = nodes_df.loc[nodes_df["label"] == "Collector", "name"].nunique()
print(f"Graph built from method='{GRAPH_METHOD}'.")
print(f"Nodes: {nodes_df.shape[0]} | Edges: {edges_df.shape[0]}")
print(f"Collector entity resolution: {n_collectors_before} verbatim → {n_collectors_after} canonical")
display(nodes_df.head(8))
display(edges_df.head(8))


In [ ]:
# ============================================================
# 19. Final markdown report
# ============================================================
# Everything in this report is computed live from the dataframes built above.
# Nothing is hardcoded so the report tracks reality across reruns.
# Structure follows what an NHM panel would want to see in order:
#   1. What we ran on  (sample composition, honesty)
#   2. How OCR did     (proxy metrics, layout fallback rate)
#   3. How extraction did  (per-field, per-method, with CIs)
#   4. Where it fails  (stratified by OCR quality)
#   5. What reconciliation added
#   6. Limitations stated explicitly

def fmt_pct(x):
    if pd.isna(x): return "—"
    return f"{100*x:.1f}%"

def md_table(df, **kwargs):
    if df is None or len(df) == 0:
        return "_(empty)_\n"
    return df.to_markdown(index=False, **kwargs) + "\n"

report = []
A = report.append
A("# HerbariumScribe v2 — Report\n\n")
A(f"_Generated: {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}_\n\n")
A(f"**Data source:** {data_source}\n\n")
A(f"**LLM backend:** `{LLM_BACKEND}` "
  f"(`{QWEN_MODEL_ID}` / `{ANTHROPIC_MODEL}`)\n\n")
A(f"**Layout tier order:** Hespi → PPStructure → geometric  "
  f"(Hespi available: `{USE_HESPI}`, PPStructure available: `{USE_PPSTRUCTURE}`)\n\n")
A(f"**OCR:** PaddleOCR `{OCR_PRIMARY_LANG}` primary; Tesseract fallback "
  f"(< {PADDLE_LOW_CONF_THRESHOLD} confidence)\n\n")
A(f"**Reconciliation:** GBIF /species/match primary; cosine over local subset "
  f"(thresholds {COS_AUTO_REPLACE} auto-replace, {COS_FLAG_REVIEW} flag-for-review)\n\n")

# 1. Sample composition
A("---\n\n## 1. Sample composition\n\n")
A(f"- DEMO_SET: {len(demo_set)} records (used ONLY as RAG few-shots)\n")
A(f"- EVAL_SET: {len(eval_set)} records (used ONLY for evaluation)\n")
A(f"- Overlap by occurrenceID: 0 (asserted)\n")
A(f"- Institutions in DEMO: {demo_set['institutionCode'].nunique()}; "
  f"in EVAL: {eval_set['institutionCode'].nunique()}; "
  f"intersection: {len(set(demo_set['institutionCode']) & set(eval_set['institutionCode']))}\n\n")
if demo_overlap_warning:
    A(f"> **{demo_overlap_warning}**\n\n")
else:
    A("> Institution-disjoint split achieved.\n\n")
A("Per-institution counts:\n\n")
A(md_table(inst_summary.reset_index().rename(columns={"index": "institutionCode"})))
A(f"\nImages successfully downloaded: DEMO {demo_ok}/{len(demo_set)}, EVAL {eval_ok}/{len(eval_set)}.\n\n")

# 2. OCR quality
A("---\n\n## 2. OCR proxy quality (transcription proxy, **not** CER/WER)\n\n")
A("Two proxies are reported. `literal_present` is the strict substring "
  "presence of the gold value in OCR text (after lowercasing). "
  "`normalised_score` is `rapidfuzz.partial_ratio` ∈ [0,1].\n\n")
A("**Important — see Limitations §6.** These are proxies because the Dillen 2019 "
  "benchmark ships curator-normalised Darwin Core metadata, NOT literal label "
  "transcriptions. We DO NOT report CER or WER.\n\n")
if len(ocr_presence_df):
    ocr_summary = (ocr_presence_df.groupby("field")
                   .agg(literal_present=("literal_present", "mean"),
                        normalised_score=("normalised_score", "mean"),
                        n_records=("normalised_score", "count"))
                   .round(3).reset_index())
    A(md_table(ocr_summary))
A(f"\nOCR-quality tertile distribution in EVAL_SET: "
  f"{dict(eval_set['ocr_tertile'].value_counts())}\n\n")

# Layout fallback rate
A("Layout detector tier distribution across all images:\n\n")
tier_rows = [{"tier": k, "n": v, "pct": fmt_pct(v/max(len(layout_rows),1))}
             for k, v in fallback_counter.items()]
A(md_table(pd.DataFrame(tier_rows)))

# OCR engine choice distribution
if len(ocr_region_df):
    A("\nOCR engine selected per crop:\n\n")
    eng_dist = (ocr_region_df["engine_used"].value_counts()
                .rename_axis("engine").reset_index(name="count"))
    eng_dist["pct"] = (100 * eng_dist["count"] / eng_dist["count"].sum()).round(1)
    A(md_table(eng_dist))

# 3. Extraction metrics
A("---\n\n## 3. Structured extraction metrics\n\n")
A("### 3a. Headline per-method (verbatim, bootstrap 95% CI on mean per-record token-F1)\n\n")
A(md_table(method_summary.round(3)))
A("\nThe CI bands matter: N=50 with stratified denominators is still small. "
  "If two methods' CIs overlap, do not claim a difference between them on "
  "this run alone.\n\n")

A("### 3b. Per-field × per-method (verbatim)\n\n")
A(md_table(eval_verbatim.round(3)))

A("\n### 3c. Per-field × per-method (canonical, after GBIF + GeoNames reconciliation)\n\n")
A("Use this view to estimate the **lift from reconciliation**. "
  "Where a method's `scientificName` / `country` / `stateProvince` improves "
  "in the canonical table vs the verbatim table, the gain came from authority-file grounding.\n\n")
A(md_table(eval_canonical.round(3)))

A("\n### 3d. Stratified by OCR-quality tertile\n\n")
A("This isolates 'is OCR the bottleneck' from 'is extraction the bottleneck'. "
  "If a method's F1 collapses in the `low` tertile but is acceptable in `high`, "
  "the upstream OCR is the constraint; if it's flat across tertiles, the "
  "extractor itself is the constraint.\n\n")
A(md_table(eval_stratified.round(3)))

# 4. Reconciliation outcomes
A("---\n\n## 4. Reconciliation outcomes\n\n")
A("### 4a. GBIF /species/match types\n\n")
A(md_table(recon_df["scientificName_match_type"].value_counts()
           .rename_axis("match_type").reset_index(name="n")))

A("\n### 4b. GBIF subset coverage of EVAL gold genera\n\n")
A(f"Fixed GBIF subset covers **{genus_coverage} / {len(gold_genera)}** "
  f"EVAL gold genera ({100*genus_coverage/max(len(gold_genera),1):.1f}%). "
  f"This is the Q1(c) honest reporting: the index is declared in advance, "
  f"not derived from the eval set.\n\n")
if len(gold_genera) - genus_coverage:
    uncovered = [g for g in gold_genera if g not in covered_genera]
    A(f"Uncovered EVAL genera ({len(uncovered)}): " +
      ", ".join(uncovered[:20]) +
      ("..." if len(uncovered) > 20 else "") + "\n\n")

A("\n### 4c. Country reconciliation methods\n\n")
A(md_table(recon_df["country_method"].value_counts()
           .rename_axis("method").reset_index(name="n")))

# Synonym flags
n_syn = int(recon_df["scientificName_is_synonym"].sum())
A(f"\n**Synonym flags surfaced:** {n_syn} extractions where GBIF resolved the "
  f"verbatim name to a synonym. Per Q6 policy, the verbatim name is preserved; "
  f"the canonical accepted name is added; a `synonym_match` warning is attached.\n\n")

# 5. Graph
A("---\n\n## 5. Graph export\n\n")
A(f"Built from method `{GRAPH_METHOD}` (canonical scientificName + canonical country "
  f"+ cosine-clustered recordedBy).\n\n")
A(f"- Nodes: {nodes_df.shape[0]} ({nodes_df['label'].value_counts().to_dict()})\n")
A(f"- Edges: {edges_df.shape[0]} ({edges_df['relationship'].value_counts().to_dict()})\n")
A(f"- Collector entity resolution: {n_collectors_before} verbatim → {n_collectors_after} canonical "
  f"(merged at cosine ≥ 0.92 onto longest variant)\n\n")

# 6. Limitations
A("---\n\n## 6. Limitations — read before defending numbers\n\n")
A("1. **Gold is catalogue metadata, not label transcription.** The Dillen et al. "
  "(2019) benchmark provides curator-normalised Darwin Core records — not "
  "the literal text on the label. So a perfect extraction of `\"G. Forrest\"` "
  "matches a gold value of `\"George Forrest\"` only after our string normalisation. "
  "What we measure is *normalised structured extraction*, not OCR CER/WER. "
  "CER/WER claims would require manual label transcriptions that this benchmark does not ship.\n\n")
A("2. **OCR proxy metrics ≠ OCR quality.** `literal_present` and `normalised_score` "
  "tell us whether the gold value *appears in OCR text*. They cannot detect "
  "an OCR engine that produces fluent but wrong text around the right value.\n\n")
A("3. **N=50 with stratified denominators gives wide CIs.** Per-cell denominators "
  "in the tertile-stratified tables are often 5–15 records. Do not over-interpret "
  "per-cell deltas; trust the headline CI bands.\n\n")
A("4. **GBIF subset is European-flora biased by construction.** This is the Q1(c) "
  "decision: the index is fixed in advance and coverage of EVAL genera is reported. "
  "A specimen from a tropical family outside the index will fall back to the "
  "cosine path or to NONE — that is honest behaviour, not a bug.\n\n")
A("5. **Layout Tier 1 (Hespi) is trained on University of Melbourne herbaria.** "
  "European herbarium label layouts may differ; fallback rate to Tier 2/3 is "
  "reported above.\n\n")
A("6. **No fine-tuning was performed in this notebook.** All models are pretrained.\n\n")
A("7. **`recordedBy` entity resolution uses cosine ≥ 0.92 over MiniLM.** This will "
  "miss heavy abbreviations (e.g. `'GF'` vs `'George Forrest'`) and may over-merge "
  "common surnames (`'Smith'`). Bionomia/Wikidata integration would resolve this "
  "but is out of scope for v2.\n\n")

report_md = "".join(report)
report_path = REPORT_DIR / "v2_report.md"
report_path.write_text(report_md, encoding="utf-8")
display(Markdown(report_md))
print(f"\nSaved report: {report_path}")


In [ ]:
# ============================================================
# 20. Package outputs
# ============================================================
zip_path = shutil.make_archive("/content/herbarium_scribe_v2_outputs", "zip", ROOT)
print("Created:", zip_path)

print("\nKey outputs (all checkpointed for partial rerun):")
key_outputs = [
    # Splits
    PROCESSED_DIR / "demo_set.csv",
    PROCESSED_DIR / "eval_set.csv",
    PROCESSED_DIR / "split_institution_summary.csv",
    # Layout + OCR
    PROCESSED_DIR / "layout_boxes.jsonl",
    PROCESSED_DIR / "ocr_by_region.csv",
    PROCESSED_DIR / "ocr_combined.csv",
    PROCESSED_DIR / "ocr_proxy_field_presence.csv",
    # Authority
    AUTHORITY_DIR / "gbif_subset.csv",
    AUTHORITY_DIR / "geonames_subset.csv",
    AUTHORITY_DIR / "gbif_match_cache.json",
    # Extractions
    PROCESSED_DIR / "extractions.jsonl",
    PROCESSED_DIR / "extractions_flat.csv",
    PROCESSED_DIR / "extractions_flat_reconciled.csv",
    PROCESSED_DIR / "rag_retrieval_log.csv",
    PROCESSED_DIR / "reconciliation.csv",
    # Eval
    PROCESSED_DIR / "eval_verbatim.csv",
    PROCESSED_DIR / "eval_canonical.csv",
    PROCESSED_DIR / "eval_stratified_by_ocr_tertile.csv",
    PROCESSED_DIR / "method_summary_with_ci.csv",
    # Graph
    PROCESSED_DIR / "graph_nodes.csv",
    PROCESSED_DIR / "graph_edges.csv",
    # Report + logs
    REPORT_DIR / "v2_report.md",
    STAGE_LOG_PATH,
]
for p in key_outputs:
    exists = "✓" if Path(p).exists() else "✗"
    print(f"  {exists} {p}")

print("\nTo rerun a single stage: delete its CSV/JSONL and rerun that cell.")
print("To resume extraction after a crash: just re-run cell 15 — JSONL is appended-to, with skip-if-done.")


## Reproducibility checklist

- `random_state = 42` everywhere a sample is drawn.
- DEMO_SET and EVAL_SET asserted disjoint at split time.
- GBIF subset is **fixed in advance** (declared in cell 2); coverage of EVAL gold genera is reported, not engineered.
- All authority files (GBIF subset, GeoNames subset, embeddings, GBIF /species/match responses) are cached on disk; reruns are fast.
- Every stage writes a CSV or JSONL checkpoint. Delete one and the cell re-runs from scratch; leave it and the cell short-circuits.
- LLM backend is one variable + one env var to flip. Both the local Qwen path and the API path go through `call_llm(messages) -> str`.
- The report's numbers are computed live from the dataframes; nothing is hardcoded.

## How to defend the choices in interview

| Choice | One-line defense |
| --- | --- |
| Hespi as layout Tier 1 | Domain-trained on ~4k annotated herbarium sheets (Turnbull et al. 2024, *BioScience*); reported mAP 0.97 on the institutional-label class. Apache 2.0, weights auto-download. |
| PaddleOCR `latin` script | Single most multilingual OCR engine for Latin-alphabet European labels; second-opinion Tesseract handles the long tail Paddle gets wrong. |
| `all-MiniLM-L6-v2` | 384-dim, 80MB, CPU-fast — fits alongside Qwen 7B in T4 VRAM; standard baseline for English bio/geo NER retrieval. |
| Fixed GBIF subset (Q1c) | Index declared in advance, coverage reported. Deriving from EVAL gold would guarantee 100% coverage by construction and leak. |
| GBIF `/species/match` primary | Typed match enum + accepted-name resolution baked in; far more defensible than a single cosine cutoff. |
| Cosine fallback at 0.92 / 0.75 | Reasoned starting points; the report prints the actual cosine distribution per run so they can be retuned with evidence rather than vibes. |
| `scientificName_verbatim` + `_canonical` (Q6) | Never silently overwrite curatorial input; surface synonyms with a warning instead. DiSSCo-aligned. |
| Bootstrap CI on per-record mean F1 | Bootstrap unit is the record, not the (record × field) cell — otherwise CIs are anti-conservative. |
| Stratified random sampling on `institutionCode` | `head(N)` on the Zenodo CSV is single-institution; stratification + seed=42 makes the eval composition reproducible *and* representative. |
| Institution-disjoint DEMO/EVAL where possible (H3) | Avoids leaking institutional formatting conventions through few-shots; falls back with a loud warning if the data doesn't allow it. |
